In [1]:
import pandas as pd
import numpy as np
import os

# load master data - takes 3 seconds now
master_df = pd.read_parquet(r"D:\CricMetric-AI\data\processed\master_odi.parquet")

print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")

Shape: (1349408, 30)
Columns: ['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year']


In [2]:
cols_to_drop = [
    'extra1', 'extra2', 'extra3',
    'running_total', 'penalty',
    'other_wicket_type', 'other_player_dismissed'
]
cols_to_drop = [col for col in cols_to_drop if col in master_df.columns]
master_df = master_df.drop(columns=cols_to_drop)

print(f"Remaining columns: {master_df.shape[1]}")
print(master_df.columns.tolist())

Remaining columns: 30
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year']


In [3]:
# check null values in each column
null_counts = master_df.isnull().sum()
null_percent = (null_counts / len(master_df) * 100).round(2)

null_df = pd.DataFrame({
    'null_count': null_counts,
    'null_percent': null_percent
})

# only show columns that have nulls
null_df = null_df[null_df['null_count'] > 0]
print(null_df)

Empty DataFrame
Columns: [null_count, null_percent]
Index: []


In [4]:
# check for duplicate balls (same match, same over, same ball number)
duplicates = master_df.duplicated(
    subset=['match_id', 'over_num', 'ball_num', 'innings']
).sum()

print(f"Duplicate balls: {duplicates}")

Duplicate balls: 0


In [5]:

# every match → every innings → every over → every ball in order
master_df = master_df.sort_values(
    ['match_id', 'innings', 'over_num', 'ball_num']
).reset_index(drop=True)

print("Sorted successfully")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 'batter', 'runs_off_bat']].head(10))

Sorted successfully
  match_id  innings  over_num  ball_num     batter  runs_off_bat
0  1000887        1         0         1  DA Warner             0
1  1000887        1         0         2  DA Warner             0
2  1000887        1         0         3  DA Warner             0
3  1000887        1         0         4  DA Warner             0
4  1000887        1         0         5  DA Warner             0
5  1000887        1         0         6  DA Warner             0
6  1000887        1         0         7  DA Warner             0
7  1000887        1         1         1    TM Head             0
8  1000887        1         1         2    TM Head             1
9  1000887        1         1         3  DA Warner             0


In [6]:
print(master_df['wicket_type'].value_counts().head(20))
print("\nUnique values sample:")
print(master_df['wicket_type'].unique()[:20])

wicket_type
""                       1312494
caught                     21118
bowled                      6915
lbw                         4102
run out                     2690
caught and bowled           1108
stumped                      882
retired hurt                  57
hit wicket                    35
obstructing the field          6
"caught                        1
Name: count, dtype: int64

Unique values sample:
<ArrowStringArray>
[                   '""',                'bowled',                'caught',
               'run out',                   'lbw',          'retired hurt',
               'stumped',     'caught and bowled',            'hit wicket',
 'obstructing the field',               '"caught']
Length: 11, dtype: str


In [7]:
# flag each ball as wicket or not
# recompute is_wicket correctly after sorting
master_df['is_wicket'] = master_df['wicket_type'].apply(
    lambda x: 1 if x not in ['', '""'] else 0
)

# verify - should show mostly 0s with occasional 1s
print("Wicket distribution:")
print(master_df['is_wicket'].value_counts())

# recompute cumulative wickets
master_df['cumulative_wickets'] = master_df.groupby(
    ['match_id', 'innings']
)['is_wicket'].cumsum()

# check first match again
print("\nFirst 15 balls:")
print(master_df[['match_id', 'innings', 'over_num', 'ball_num',
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))


Wicket distribution:
is_wicket
0    1312494
1      36914
Name: count, dtype: int64

First 15 balls:
   match_id  innings  over_num  ball_num     batter  runs_off_bat  is_wicket  \
0   1000887        1         0         1  DA Warner             0          0   
1   1000887        1         0         2  DA Warner             0          0   
2   1000887        1         0         3  DA Warner             0          0   
3   1000887        1         0         4  DA Warner             0          0   
4   1000887        1         0         5  DA Warner             0          0   
5   1000887        1         0         6  DA Warner             0          0   
6   1000887        1         0         7  DA Warner             0          0   
7   1000887        1         1         1    TM Head             0          0   
8   1000887        1         1         2    TM Head             1          0   
9   1000887        1         1         3  DA Warner             0          0   
10  1000887        1

In [8]:
# compute total runs per ball (batting runs + extras)
master_df['total_runs_ball'] = master_df['runs_off_bat'] + master_df['extras']

# cumulative runs and wickets within each innings of each match
master_df['cumulative_runs'] = master_df.groupby(
    ['match_id', 'innings']
)['total_runs_ball'].cumsum()

In [9]:

# verify
print(master_df[['match_id', 'innings', 'over_num', 'ball_num', 
                  'batter', 'runs_off_bat', 'is_wicket',
                  'cumulative_runs', 'cumulative_wickets']].head(15))

   match_id  innings  over_num  ball_num     batter  runs_off_bat  is_wicket  \
0   1000887        1         0         1  DA Warner             0          0   
1   1000887        1         0         2  DA Warner             0          0   
2   1000887        1         0         3  DA Warner             0          0   
3   1000887        1         0         4  DA Warner             0          0   
4   1000887        1         0         5  DA Warner             0          0   
5   1000887        1         0         6  DA Warner             0          0   
6   1000887        1         0         7  DA Warner             0          0   
7   1000887        1         1         1    TM Head             0          0   
8   1000887        1         1         2    TM Head             1          0   
9   1000887        1         1         3  DA Warner             0          0   
10  1000887        1         1         4  DA Warner             0          0   
11  1000887        1         1         5

In [10]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 30 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   innings             1349408 non-null  int64         
 1   over                1349408 non-null  str           
 2   batting_team        1349408 non-null  str           
 3   batter              1349408 non-null  str           
 4   non_striker         1349408 non-null  str           
 5   bowler              1349408 non-null  str           
 6   runs_off_bat        1349408 non-null  int64         
 7   extras              1349408 non-null  int64         
 8   wides               1349408 non-null  int64         
 9   noballs             1349408 non-null  int64         
 10  byes                1349408 non-null  int64         
 11  legbyes             1349408 non-null  int64         
 12  wicket_type         1349408 non-null  str           
 13  player_dismissed    134

In [11]:
# string to int conversions
int_cols = ['innings', 'over_num', 'ball_num', 'byes', 'legbyes']

for col in int_cols:
    master_df[col] = pd.to_numeric(master_df[col], errors='coerce').fillna(0).astype(int)

# match_date to datetime
master_df['match_date'] = pd.to_datetime(master_df['match_date'], errors='coerce')

# extract year as separate column
master_df['year'] = master_df['match_date'].dt.year

master_df.info()



<class 'pandas.DataFrame'>
RangeIndex: 1349408 entries, 0 to 1349407
Data columns (total 30 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   innings             1349408 non-null  int64         
 1   over                1349408 non-null  str           
 2   batting_team        1349408 non-null  str           
 3   batter              1349408 non-null  str           
 4   non_striker         1349408 non-null  str           
 5   bowler              1349408 non-null  str           
 6   runs_off_bat        1349408 non-null  int64         
 7   extras              1349408 non-null  int64         
 8   wides               1349408 non-null  int64         
 9   noballs             1349408 non-null  int64         
 10  byes                1349408 non-null  int64         
 11  legbyes             1349408 non-null  int64         
 12  wicket_type         1349408 non-null  str           
 13  player_dismissed    134

In [12]:
master_df.to_parquet(
    r"D:\CricMetric-AI\data\processed\master_odi.parquet",
    index=False
)
print("Saved cleaned dataframe successfully")
print(f"Shape: {master_df.shape}")

Saved cleaned dataframe successfully
Shape: (1349408, 30)


## Situation 1 — Early Collapse (PPI Component 1)

### What this measures
How ALL batters perform when team is in trouble early in the innings.
Specifically: every ball faced when 3+ wickets have already fallen
in first 20 overs of innings 1.

### Why innings 1 only
In innings 2 (chase), strike rate is context-dependent.
A batter scoring at 60 SR when required run rate is 5 is fine.
But in innings 1, any collapse situation requires scoring —
no free passes. SR is the right and only metric for innings 1.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| cumulative_wickets | >= 3 | Team in collapse |
| over_num | < 20 | Early innings only |
| innings | == 1 | First innings pressure only |
| wides | == 0 | Legal deliveries only |

### Who gets counted
Both new batters AND set batters are counted.

Example:
- Team is 2 wickets down, Rohit batting, Kohli non-striker
- Kohli gets out → 3rd wicket falls → Dhoni walks in
- Ball after wicket: Dhoni facing → counted under Dhoni ✓
- Next ball: Rohit facing → counted under Rohit ✓
- Both are under pressure regardless of how long they batted

Non-striker is NEVER counted — they are not facing the ball.
Only the batter column (striker) gets credited.

### Design decision: counting all batters in pressure situation
Both new batters AND set batters are counted when
cumulative_wickets >= 3.
Reason: pressure is situational — any batter faces increased
mental and tactical pressure when team loses 3+ wickets,
regardless of how long they have been batting.

### Metric used
Strike Rate under pressure = (runs / balls faced) × 100

In early collapse, scoring speed matters:
- A batter who scores at 80 SR stabilizes AND accelerates innings
- SR naturally punishes batters who get out cheaply
- SR naturally rewards batters who score despite pressure
- No separate dismissal count needed — already baked into SR

### Why SR and not average here
Average = runs per dismissal
SR = runs per ball

In collapse situations, BOTH survival and scoring speed matter.
But SR captures both better because:
- Getting out cheaply → low runs, same balls → low SR
- Surviving but not scoring → many balls, few runs → low SR
- Scoring quickly → many runs, fewer balls → high SR

### Minimum qualification
61+ balls faced in pressure situations.
Threshold = population average (60.9 balls per batter).
Statistically justified — batter must have faced at least
the average pressure exposure to qualify.
More defensible than arbitrary cutoff like 50.


In [13]:
# Situation 1: batting when 3+ wickets down in first 20 overs
pressure_s1 = master_df[
    (master_df['cumulative_wickets'] >= 3) &
    (master_df['over_num'] < 20) &
    (master_df['innings'] == 1) &
    (master_df['wides'] == 0)
].copy()

print(f"Pressure situation 1 balls: {len(pressure_s1)}")
print(f"Unique batters in pressure: {pressure_s1['batter'].nunique()}")

# compute stats for ALL batters first (before filtering)
s1_all = pressure_s1.groupby('batter').agg(
    s1_runs=('runs_off_bat', 'sum'),
    s1_balls=('runs_off_bat', 'count')
).reset_index()

s1_all['s1_sr'] = (
    s1_all['s1_runs'] / s1_all['s1_balls'] * 100
).round(2)

# statistical analysis BEFORE filtering
print("\nPressure balls distribution (all batters):")
print(s1_all['s1_balls'].describe())

Pressure situation 1 balls: 53595
Unique batters in pressure: 880

Pressure balls distribution (all batters):
count    880.000000
mean      60.903409
std       96.369187
min        1.000000
25%        7.000000
50%       27.000000
75%       76.000000
max      738.000000
Name: s1_balls, dtype: float64


In [14]:
# threshold = mean (population average)
min_s1_balls = int(s1_all['s1_balls'].mean())
print(f"Statistical threshold (mean): {min_s1_balls} balls")

s1_stats = s1_all[s1_all['s1_balls'] >= min_s1_balls].copy()

print(f"Batters qualifying ({min_s1_balls}+ pressure balls): {len(s1_stats)}")
print("\nTop 20 by strike rate under pressure:")
print(s1_stats.sort_values('s1_sr', ascending=False).head(20))

Statistical threshold (mean): 60 balls
Batters qualifying (60+ pressure balls): 250

Top 20 by strike rate under pressure:
                    batter  s1_runs  s1_balls   s1_sr
162            CJ Anderson      159       108  147.22
364               JD Ryder      100        78  128.21
386             JM Davison       71        60  118.33
588            NLTC Perera      107        93  115.05
501            MDKJ Perera      105        98  107.14
201              DA Warner      118       115  102.61
121            BJ McMullen       77        78   98.72
165            CJ Ferguson       72        73   98.63
518              ML Hayden       89        91   97.80
172              CL Cairns       79        86   91.86
682             RT Ponting      160       176   90.91
719                SD Hope      212       238   89.08
813             TLW Cooper       81        91   89.01
403            JS Malhotra       95       108   87.96
756          ST Jayasuriya       73        83   87.95
604  Nazmul H

In [15]:
# check specific players
check_players = ['V Kohli', 'RG Sharma', 'MS Dhoni', 'AB de Villiers', 'KC Sangakkara']

for player in check_players:
    player_data = pressure_s1[pressure_s1['batter'] == player]
    print(f"{player}: {len(player_data)} pressure balls")

V Kohli: 164 pressure balls
RG Sharma: 202 pressure balls
MS Dhoni: 531 pressure balls
AB de Villiers: 323 pressure balls
KC Sangakkara: 661 pressure balls



### Key finding
250 batters qualified with 60+ pressure balls.
Legends like Dhoni (531 balls), Sangakkara (661 balls)
faced the most pressure — reflecting their middle order roles.
Kohli (164 balls) faced less — reflecting his top order position
where team rarely collapsed before he arrived.
Even with fewer balls, Kohli's SR in pressure reflects
how well he performed when collapse DID happen around him.

### Known limitation
Current implementation counts set batters AND new batters equally.
Ideally we would distinguish between:
- New batter walking in cold during collapse (harder)
- Set batter already there when collapse happens (slightly easier)
This distinction is a future improvement opportunity.
For now, situational pressure is treated equally for all batters

In [16]:
# Situation 2: overs 40-50, chasing 270+, innings 2
# first find matches where team 2 needed 270+
innings2 = master_df[master_df['innings'] == 2].copy()

# get target for each match (max cumulative runs in innings 1 + 1)
innings1_totals = master_df[
    master_df['innings'] == 1
].groupby('match_id')['cumulative_runs'].max().reset_index()
innings1_totals.columns = ['match_id', 'innings1_total']
innings1_totals['target'] = innings1_totals['innings1_total'] + 1

# merge target into innings 2
innings2 = innings2.merge(innings1_totals, on='match_id', how='left')

In [17]:
# filter: overs 40+, target 270+, legal deliveries only
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 270) &
    (innings2['wides'] == 0)
].copy()

print(f"Pressure situation-2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

# compute stats
s2_stats = pressure_s2.groupby('batter').agg(
    s2_runs=('runs_off_bat', 'sum'),
    s2_balls=('runs_off_bat', 'count')
).reset_index()

s2_stats['s2_sr'] = (s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100).round(2)
s2_stats = s2_stats[s2_stats['s2_balls'] >= 50]

print(f"\nBatters with 30+ death chase balls: {len(s2_stats)}")
print("\nTop 10 by death chase strike rate:")
print(s2_stats.sort_values('s2_sr', ascending=False).head(10))

Pressure situation-2 balls: 33356
Unique batters: 931

Batters with 30+ death chase balls: 204

Top 10 by death chase strike rate:
             batter  s2_runs  s2_balls   s2_sr
511    MG Bracewell      151        81  186.42
496        MA Leask      110        61  180.33
814   Shahid Afridi      209       117  178.63
535      MP Stoinis      116        65  178.46
253    Fakhar Zaman      123        69  178.26
744       SB Styris      108        63  171.43
193   DAS Gunaratne      122        72  169.44
431    KP Pietersen      132        79  167.09
332  Iftikhar Ahmed       88        53  166.04
670      R Shepherd      108        66  163.64


In [18]:
print(innings2.shape)
print(innings2.columns.tolist())

(616034, 32)
['innings', 'over', 'batting_team', 'batter', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'wicket_type', 'player_dismissed', 'match_id', 'match_date', 'team1', 'team2', 'venue', 'event', 'toss_winner', 'city', 'season', 'over_num', 'ball_num', 'is_wicket', 'total_runs_ball', 'cumulative_runs', 'cumulative_wickets', 'year', 'innings1_total', 'target']


In [19]:
# Count pressure innings for each batter
pressure_innings_per_batter = (
    pressure_s2.groupby("batter")["match_id"]
    .nunique()
    .reset_index(name="pressure_innings")
)

# Average pressure innings across all batters
avg_pressure_innings = pressure_innings_per_batter["pressure_innings"].median()

print("Average pressure innings per batter:", avg_pressure_innings)

# Optional: see all batters sorted
print(
    pressure_innings_per_batter.sort_values(
        "pressure_innings",
        ascending=False
    )
)

Average pressure innings per batter: 2.0
               batter  pressure_innings
541          MS Dhoni                32
675         RA Jadeja                29
549       Mahmudullah                22
551  Mashrafe Mortaza                19
886           V Kohli                19
..                ...               ...
531        MN van Wyk                 1
528          MN Nawaz                 1
6               A Nao                 1
905   WTS Porterfield                 1
903       WK McCallan                 1

[931 rows x 2 columns]


In [20]:
pressure_innings_per_batter["pressure_innings"].describe()

count    931.000000
mean       3.243824
std        3.393170
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       32.000000
Name: pressure_innings, dtype: float64

In [21]:
threshold = pressure_innings_per_batter["pressure_innings"].quantile(0.75)

print(threshold)

4.0


In [22]:
match_outcomes = innings2.groupby('match_id').agg(
    chasing_team=('batting_team', 'first'),
    final_runs=('cumulative_runs', 'max'),
    target=('target', 'first')
).reset_index()

# did chasing team win?
match_outcomes['chase_won'] = (
    match_outcomes['final_runs'] >= match_outcomes['target']
).astype(int)

print(f"Total chase matches: {len(match_outcomes)}")
print(f"Chases won: {match_outcomes['chase_won'].sum()}")
print(f"Chase win rate: {match_outcomes['chase_won'].mean()*100:.1f}%")

# updated filter - 250+ target
pressure_s2 = innings2[
    (innings2['over_num'] >= 40) &
    (innings2['target'] >= 250) &
    (innings2['wides'] == 0)
].copy()

print(f"\nPressure situation-2 balls: {len(pressure_s2)}")
print(f"Unique batters: {pressure_s2['batter'].nunique()}")

Total chase matches: 2477
Chases won: 1180
Chase win rate: 47.6%

Pressure situation-2 balls: 42108
Unique batters: 1016


In [23]:
# merge win result into pressure_s2
pressure_s2 = pressure_s2.merge(
    match_outcomes[['match_id', 'chase_won']],
    on='match_id',
    how='left'
)

# per batter per match death chase stats
s2_innings = pressure_s2.groupby(
    ['batter', 'match_id']
).agg(
    death_runs=('runs_off_bat', 'sum'),
    death_balls=('runs_off_bat', 'count'),
    chase_won=('chase_won', 'first')
).reset_index()

# per batter overall death chase stats
s2_stats = s2_innings.groupby('batter').agg(
    s2_innings=('match_id', 'count'),
    s2_runs=('death_runs', 'sum'),
    s2_balls=('death_balls', 'sum'),
    s2_wins=('chase_won', 'sum')
).reset_index()

In [24]:
# compute metrics
s2_stats['s2_sr'] = (
    s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100
).round(2)

s2_stats['s2_win_pct'] = (
    s2_stats['s2_wins'] / s2_stats['s2_innings'] * 100
).round(2)

# combined score
s2_stats['s2_score'] = (
    s2_stats['s2_sr'] * s2_stats['s2_win_pct'] / 100
).round(2)

s2_stats = s2_stats[s2_stats['s2_innings'] >= threshold]

print(f"\nBatters with {threshold:.0f}+ death chase innings: {len(s2_stats)}")
print("\nTop 10 by combined death chase score:")
print(s2_stats.sort_values('s2_score', ascending=False)[
    ['batter', 's2_innings', 's2_runs', 's2_sr', 's2_win_pct', 's2_score']
].head(10))


Batters with 4+ death chase innings: 348

Top 10 by combined death chase score:
              batter  s2_innings  s2_runs   s2_sr  s2_win_pct  s2_score
117          BC Lara           5       45  150.00      100.00    150.00
178         CL White           4       84  144.83      100.00    144.83
467       KJ O'Brien           7      151  160.64       85.71    137.68
153  C de Grandhomme           6      123  161.84       83.33    134.86
365   Iftikhar Ahmed           4       88  166.04       75.00    124.53
330      HM Nicholls           6       81  147.27       83.33    122.72
966          V Kohli          24      516  146.18       83.33    121.81
225      DJ Mitchell           6      117  140.96       83.33    117.46
469         KL Rahul           8      149  134.23       87.50    117.45
281        G Gambhir           4       57  116.33      100.00    116.33


In [25]:
# 1. what is threshold value
print(f"Threshold value: {threshold}")

# 2. check Lara's data
lara = s2_innings[s2_innings['batter'] == 'BC Lara']
print(lara)

Threshold value: 4.0
      batter match_id  death_runs  death_balls  chase_won
423  BC Lara   249758           1            3          1
424  BC Lara   256610           0            1          1
425  BC Lara   267708           6            3          1
426  BC Lara    64865          35           17          1
427  BC Lara    66309           3            6          1


In [26]:
# statistical analysis of death_balls per innings
print("Death balls per innings distribution:")
print(s2_innings['death_balls'].describe())
print(f"\nMedian: {s2_innings['death_balls'].median()}")

# statistical analysis of innings per batter
print("\n\nInnings per batter distribution:")
innings_per_batter = s2_innings.groupby('batter')['match_id'].count()
print(innings_per_batter.describe())
print(f"\nMedian: {innings_per_batter.median()}")

Death balls per innings distribution:
count    3808.000000
mean       11.057773
std         8.467110
min         1.000000
25%         4.000000
50%         9.000000
75%        16.000000
max        48.000000
Name: death_balls, dtype: float64

Median: 9.0


Innings per batter distribution:
count    1016.000000
mean        3.748031
std         4.099921
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        40.000000
Name: match_id, dtype: float64

Median: 2.0


## Situation 2 — Death Chase (PPI Component 2)

### What this measures
How a batter performs in the final overs of a high-pressure chase.
Specifically: balls faced in overs 40-50 when chasing a target of 250+,
measuring both scoring speed AND match-winning contribution.

### Why this situation matters
Any batter can score in a low-pressure chase of 180.
The truly great finishers score fast AND win matches
when target is 250+ and every ball counts.
This separates genuine death specialists from comfortable scorers.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| over_num | >= 40 | Death overs only |
| target | >= 250 | High pressure chase only |
| wides | == 0 | Legal deliveries only |
| death_balls per innings | >= 9 (median) | Genuine death batting only |

### Why target 250+ specifically
Below 250, death over batting is relatively low pressure.
Team can afford to lose wickets and still win.
Above 250, every single ball in overs 40-50 is critical.
250 represents the inflection point in ODI pressure analysis.

### Why NOT strike rate alone
Initial approach used strike rate only.
Problem identified: high SR in a losing chase means nothing.

Example:
- Batter scores 60 off 30 balls (SR 200) but team loses
- vs batter scores 40 off 28 balls (SR 143) and team wins
- Second batter is more valuable despite lower SR

### Metric evolution
| Version | Metric | Problem |
|---|---|---|
| V1 | Strike rate only | Rewarded losing contributions |
| V2 | SR × Win% | Better but small sample issues |
| V3 | SR × Win% with statistical thresholds | Current — most robust |

### Final metric: Death Chase Score = SR × Win% / 100
| Component | What it measures |
|---|---|
| s2_sr | How fast did they score in death overs? |
| s2_win_pct | How often did team win those matches? |
| s2_score | Combined — fast scoring that led to wins |

### Statistically derived thresholds
All thresholds computed from actual data distribution.
No arbitrary values used.

| Threshold | Method | Value | Reason |
|---|---|---|---|
| Min balls per innings | Median (50th percentile) | 9 balls | Removes bottom half low-exposure innings |
| Min qualifying innings | 75th percentile | 5 innings | Only top 25% by innings count qualify |

### Why median for balls, 75th percentile for innings
Balls per innings → median:
- Half of all innings had less than 9 death balls
- These are not genuine death batting situations
- Median threshold removes casual appearances

Qualifying innings → 75th percentile:
- 75% of batters had fewer than 5 qualifying innings
- Only top 25% qualify as genuine death specialists
- Ensures statistical reliability of win percentage

### Filter evolution — fixing BC Lara problem
Original filter (minimum 5 innings count only) had a flaw.

BC Lara example:
- 5 innings but faced only 1, 3, 3, 6, 17 balls per innings
- All 5 matches won → 100% win rate → inflated s2_score
- Not a genuine death batter — just happened to be on field

Fix: minimum 9 balls per qualifying innings (median threshold)
Lara's 1, 3, 3, 6 ball innings all removed
Only his 17-ball innings remains → below 5 innings minimum
Lara correctly excluded from death chase specialists


In [27]:
# statistically derived thresholds
min_balls_per_innings = int(s2_innings['death_balls'].median())
min_qualifying_innings = int(
    s2_innings.groupby('batter')['match_id'].count().quantile(0.75)
)

print(f"Minimum balls per innings (median): {min_balls_per_innings}")
print(f"Minimum qualifying innings (75th percentile): {min_qualifying_innings}")

# apply minimum balls per innings filter
s2_innings_filtered = s2_innings[
    s2_innings['death_balls'] >= min_balls_per_innings
]

print(f"Innings with {min_balls_per_innings}+ death balls: {len(s2_innings_filtered)}")

# per batter stats
s2_stats = s2_innings_filtered.groupby('batter').agg(
    s2_innings=('match_id', 'count'),
    s2_runs=('death_runs', 'sum'),
    s2_balls=('death_balls', 'sum'),
    s2_wins=('chase_won', 'sum')
).reset_index()

# metrics
s2_stats['s2_sr'] = (
    s2_stats['s2_runs'] / s2_stats['s2_balls'] * 100
).round(2)

s2_stats['s2_win_pct'] = (
    s2_stats['s2_wins'] / s2_stats['s2_innings'] * 100
).round(2)

s2_stats['s2_score'] = (
    s2_stats['s2_sr'] * s2_stats['s2_win_pct'] / 100
).round(2)

# apply minimum qualifying innings filter
s2_stats = s2_stats[s2_stats['s2_innings'] >= min_qualifying_innings]

print(f"Batters with {min_qualifying_innings}+ qualifying innings: {len(s2_stats)}")
print("\nTop 10 by death chase score:")
print(s2_stats.sort_values('s2_score', ascending=False)[
    ['batter', 's2_innings', 's2_runs',
     's2_sr', 's2_win_pct', 's2_score']
].head(10))

Minimum balls per innings (median): 9
Minimum qualifying innings (75th percentile): 5
Innings with 9+ death balls: 1965
Batters with 5+ qualifying innings: 106

Top 10 by death chase score:
            batter  s2_innings  s2_runs   s2_sr  s2_win_pct  s2_score
346     KJ O'Brien           6      151  162.37       83.33    135.30
708        V Kohli          20      501  148.22       85.00    125.99
54    Abdul Razzaq           9      283  156.35       77.78    121.61
610       SK Raina          14      370  130.74       92.86    121.41
293        JE Root           8      215  135.22       87.50    118.32
196     EJG Morgan          11      269  143.85       81.82    117.70
291    JDS Neesham           5      124  137.78       80.00    110.22
348       KL Rahul           5      139  136.27       80.00    109.02
355  KS Williamson           5       93  104.49      100.00    104.49
340  KC Sangakkara           9      218  133.74       77.78    104.02


In [28]:
import glob as gl

WC_PATH = r"D:\CricMetric-AI\data\raw\worldcup"
wc_files = gl.glob(os.path.join(WC_PATH, "*.csv"))

wc_match_ids = [
    os.path.basename(f).replace('.csv', '')
    for f in wc_files
]

print(f"World Cup match IDs loaded: {len(wc_match_ids)}")

World Cup match IDs loaded: 265


## Situation 3 — World Cup Performance (PPI Component 3)

### What this measures
How a batter performs specifically in ICC Cricket World Cup matches.
World Cup = highest pressure tournament in ODI cricket.
Every match carries national expectations, legacy implications,
and knockout consequences that bilateral series never replicate.

### Why World Cup deserves its own situation
Pressure in World Cup is categorically different from regular ODIs:
- Elimination pressure — lose and go home
- National expectations — billions watching
- Legacy defining — careers remembered by World Cup moments
- Best players from every nation at their peak simultaneously

A batter averaging 45 in bilateral series but 60 in World Cups
is fundamentally more valuable than one who does the reverse.

### Data source
265 ICC Men's Cricket World Cup matches from Cricsheet.
Match IDs extracted from separate World Cup CSV dataset.
Cross-referenced with main ODI dataset to get ball-by-ball detail.

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| match_id | in wc_match_ids | World Cup matches only |
| wides | == 0 | Legal deliveries only |

### Why NOT strike rate for World Cup
Unlike death chases, World Cup innings have varying contexts:
- Chasing 180 comfortably → low SR acceptable
- Defending 320 → anchor role more valuable than aggression
- What matters = did they score consistently AND significantly?

Strike rate would unfairly punish anchors and reward sloggers
in matches that were already won comfortably.

### Metrics used
| Metric | Formula | What it captures |
|---|---|---|
| s3_avg | total_runs / dismissals | Scoring volume per innings |
| s3_consistency | % innings with 30+ runs | Reliability across matches |
| s3_50plus | count of 50+ innings | Match-defining contributions |


In [29]:
# Situation 3: World Cup performance
pressure_s3 = master_df[
    (master_df['match_id'].isin(wc_match_ids)) &
    (master_df['wides'] == 0)
].copy()

print(f"World Cup balls: {len(pressure_s3)}")
print(f"Unique batters: {pressure_s3['batter'].nunique()}")

# runs per innings in WC
s3_innings = pressure_s3.groupby(
    ['batter', 'match_id', 'innings']
).agg(
    innings_runs=('runs_off_bat', 'sum'),
    innings_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter WC stats
s3_stats = s3_innings.groupby('batter').agg(
    s3_innings=('match_id', 'count'),
    s3_runs=('innings_runs', 'sum'),
    s3_dismissals=('got_out', 'sum'),
    s3_50plus=('innings_runs', lambda x: (x >= 50).sum())
).reset_index()

# average
s3_stats['s3_avg'] = (
    s3_stats['s3_runs'] /
    s3_stats['s3_dismissals'].replace(0, 1)
).round(2)

# consistency
s3_consistency = s3_innings.groupby('batter').apply(
    lambda x: (x['innings_runs'] >= 30).sum() / len(x) * 100
).reset_index()
s3_consistency.columns = ['batter', 's3_consistency']
s3_consistency['s3_consistency'] = s3_consistency['s3_consistency'].round(2)

s3_stats = s3_stats.merge(s3_consistency, on='batter')
# statistical analysis for S3
print("S3 innings per batter distribution:")
s3_innings_count = s3_innings.groupby('batter')['match_id'].count()
print(s3_innings_count.describe())
print(f"Median:          {s3_innings_count.median():.2f}")

World Cup balls: 137817
Unique batters: 695
S3 innings per batter distribution:
count    695.000000
mean       6.728058
std        6.058029
min        1.000000
25%        2.000000
50%        5.000000
75%        8.500000
max       35.000000
Name: match_id, dtype: float64
Median:          5.00


In [30]:
min_s3_innings = int(s3_innings_count.quantile(0.75)) + 1  
s3_stats = s3_stats[s3_stats['s3_innings'] >= min_s3_innings]

print(f"\nBatters with {min_s3_innings}+ WC innings: {len(s3_stats)}")
print("\nTop 10 by World Cup average:")
print(s3_stats.sort_values('s3_avg', ascending=False)[
    ['batter', 's3_innings', 's3_runs', 's3_avg', 
     's3_consistency', 's3_50plus']
].head(10))


Batters with 9+ WC innings: 174

Top 10 by World Cup average:
               batter  s3_innings  s3_runs  s3_avg  s3_consistency  s3_50plus
11          A Symonds          13      515   85.83           38.46          4
230          HH Gibbs          14      726   72.60           71.43          7
17     AB de Villiers          22     1207   71.00           59.09         10
429       Mahmudullah          20      894   68.77           50.00          6
512        R Ravindra           9      546   68.25           66.67          5
403         MJ Clarke          21      888   63.43           57.14          8
592           SS Iyer          10      505   63.12           60.00          5
523         RG Sharma          26     1443   62.74           69.23         12
531  RN ten Doeschate           9      435   62.14           55.56          5
342     KS Williamson          24     1055   62.06           66.67          7


In [31]:
kohli_wc = s3_stats[s3_stats['batter'] == 'V Kohli']
dhoni_wc=s3_stats[s3_stats['batter'] == 'MS Dhoni']
print(kohli_wc)
print(dhoni_wc)

      batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
664  V Kohli          35     1673             28         15   59.75   

     s3_consistency  
664           65.71  
       batter  s3_innings  s3_runs  s3_dismissals  s3_50plus  s3_avg  \
424  MS Dhoni          24      752             17          5   44.24   

     s3_consistency  
424           45.83  


## Situation 4 — Chase Recovery (PPI Component 4)

### What this measures
How a batter performs when their team loses early wickets
in a chase AND the team still wins the match.

This captures the rarest and most valuable skill in ODI batting:
the ability to rescue a chase that looked lost and win anyway.

### Why this situation is unique
S4 combines two difficult conditions simultaneously:
1. Team is in trouble (3+ wickets down early in chase)
2. Team wins despite the trouble

This means every ball in S4 represents a genuine rescue act.
The batter not only survived the collapse but contributed
enough to turn a losing position into a win.

### How S4 differs from other situations
| Situation | Context | Key difference |
|---|---|---|
| S1 | First innings collapse | Rebuild total, no target |
| S2 | Death overs chase | End game pressure, team intact |
| S3 | World Cup matches | Tournament pressure, any situation |
| S4 | Chase collapse + win | Must rescue AND win, second innings |

S4 is harder than S1 because:
- You have a target and a clock
- Every wicket lost makes target harder
- No margin for error — collapse already happened

S4 is different from S2 because:
- S2 = end game specialist (overs 40-50)
- S4 = rescue specialist (early collapse, overs 0-25)
- Different skills, different players excel in each

### Filter conditions
| Condition | Value | Reason |
|---|---|---|
| innings | == 2 | Chasing innings only |
| match_id | in won_chases | Only won chases included |
| cumulative_wickets | >= 3 | Early collapse happened |
| over_num | < 25 | Early in chase (not death overs) |
| wides | == 0 | Legal deliveries only |
| balls per innings | >= 10 (median) | Genuine recovery contribution |

### Why won_chases filter makes win% redundant
Every match in S4 is already a won chase.
Win% = 100% for all batters → useless as differentiator.
Instead we measure HOW MUCH they contributed during collapse.

### Metric: S4 Score = Average × Strike Rate / 100
| Component | Reason |
|---|---|
| s4_avg | Did they score enough runs to matter? |
| s4_sr | Did they score fast enough to keep chase alive? |
| combined | Both needed simultaneously |


### Statistically derived thresholds
Two separate thresholds applied:

**Threshold 1 — Minimum balls per qualifying innings:**
Distribution of balls per S4 innings:
| Statistic | Value |
|---|---|
| Mean | 14.95 balls |
| Median | 10.0 balls |
| 25th percentile | 1.0 ball |
| 75th percentile | 23.0 balls |

Threshold = median = 10 balls minimum per innings.
Removes bottom 50% of low-exposure recovery appearances.
A batter who faced 1-3 balls in a recovery situation
is not a genuine recovery specialist — just happened to be there.

**Threshold 2 — Minimum qualifying innings:**
Distribution of qualifying innings per batter:
| Statistic | Value |
|---|---|
| Mean | 3.56 innings |
| Median | 2.0 innings |
| 75th percentile | 4.0 innings |

Threshold = 75th percentile = 4 innings minimum.
Only top 25% by qualifying innings count.
Ensures statistical reliability of average and SR calculations.

### Filter evolution
Original: minimum 5 innings (arbitrary)
Problem 1: included innings with 1-3 balls (not genuine recovery)
Problem 2: threshold not statistically justified

Fix: two-stage statistical filter
Stage 1: remove innings with < 10 balls (median threshold)
Stage 2: require 4+ qualifying innings (75th percentile)
Result: 75 genuine chase recovery specialists identified

### Known limitation
S4 only covers chases that were eventually WON.
This means we have no data on batters who made valiant
recovery attempts in losing causes.
A batter who scored 90 in a losing chase from 30/4
gets no credit in S4 — only winners are measured.
This is intentional: we reward match-winning recoveries,
not brave but ultimately unsuccessful ones.
Future improvement: create a separate partial credit score
for recovery attempts in losing causes, weighted lower.

In [32]:
# first get matches where chase was won
# we already have match_outcomes from S2
won_chases = match_outcomes[
    match_outcomes['chase_won'] == 1
]['match_id'].tolist()

print(f"Total won chases: {len(won_chases)}")

# filter: innings 2, early collapse, won matches only
pressure_s4 = innings2[
    (innings2['cumulative_wickets'] >= 3) &
    (innings2['over_num'] < 25) &
    (innings2['match_id'].isin(won_chases)) &
    (innings2['wides'] == 0)
].copy()

print(f"S4 pressure balls: {len(pressure_s4)}")
print(f"Unique batters: {pressure_s4['batter'].nunique()}")

# runs per innings per batter in S4
s4_innings = pressure_s4.groupby(
    ['batter', 'match_id']
).agg(
    s4_runs=('runs_off_bat', 'sum'),
    s4_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# per batter overall S4 stats
s4_stats = s4_innings.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum'),
    s4_dismissals=('got_out', 'sum')
).reset_index()

# average in recovery situations
s4_stats['s4_avg'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_dismissals'].replace(0, 1)
).round(2)

# strike rate too
s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_total_balls'] * 100
).round(2)

# combined score avg × SR / 100
s4_stats['s4_score'] = (
    s4_stats['s4_avg'] * s4_stats['s4_sr'] / 100
).round(2)
# statistical analysis for S4
print("S4 innings per batter distribution:")
s4_innings_count = s4_innings.groupby('batter')['match_id'].count()
print(s4_innings_count.describe())
print(f"Median:          {s4_innings_count.median():.2f}")


Total won chases: 1180
S4 pressure balls: 29473
Unique batters: 554
S4 innings per batter distribution:
count    554.000000
mean       3.557762
std        4.155793
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       30.000000
Name: match_id, dtype: float64
Median:          2.00


In [33]:
min_s4_innings = int(s4_innings_count.quantile(0.75)) 
s4_stats = s4_stats[s4_stats['s4_innings'] >= min_s4_innings]

print(f"\nBatters with 5+ recovery innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_avg', 's4_sr', 's4_score']
].head(10))


Batters with 5+ recovery innings: 172

Top 10 by chase recovery score:
                    batter  s4_innings  s4_total_runs  s4_avg   s4_sr  \
384  Nazmul Hossain Shanto           4             98    98.0  122.50   
3               A Flintoff          11            255    85.0  118.60   
265           KIC Asalanka           8            218   109.0   86.51   
12          AB de Villiers          18            376    94.0   96.91   
260             KA Pollard           7            150    75.0  104.17   
320           MG Bracewell           5             85    85.0   90.43   
240        JN Loftie-Eaton           6             87    87.0   87.88   
268               KL Rahul           6            116   116.0   61.38   
69             BB McCullum           5             98    98.0   68.53   
350           Milind Kumar           4             63    63.0  103.28   

     s4_score  
384    120.05  
3      100.81  
265     94.30  
12      91.10  
260     78.13  
320     76.87  
240     76.4

In [34]:
# statistical analysis of balls per S4 innings
print("S4 balls per innings distribution:")
print(s4_innings['s4_balls'].describe())
print(f"Median: {s4_innings['s4_balls'].median()}")
print(f"Mean: {s4_innings['s4_balls'].mean():.2f}")

S4 balls per innings distribution:
count    1971.000000
mean       14.953323
std        15.691533
min         1.000000
25%         1.000000
50%        10.000000
75%        23.000000
max        76.000000
Name: s4_balls, dtype: float64
Median: 10.0
Mean: 14.95


In [35]:
# statistically derived thresholds for S4
min_s4_balls_per_innings = int(s4_innings['s4_balls'].median())  # = 10
min_s4_innings = int(s4_innings_count.quantile(0.75))  # = 4

print(f"S4 min balls per innings (median): {min_s4_balls_per_innings}")
print(f"S4 min qualifying innings (75th percentile): {min_s4_innings}")

# apply minimum balls per innings filter
s4_innings_filtered = s4_innings[
    s4_innings['s4_balls'] >= min_s4_balls_per_innings
]

print(f"Innings with {min_s4_balls_per_innings}+ recovery balls: {len(s4_innings_filtered)}")

# per batter stats
s4_stats = s4_innings_filtered.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum'),
    s4_dismissals=('got_out', 'sum')
).reset_index()

# metrics
s4_stats['s4_avg'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_dismissals'].replace(0, 1)
).round(2)

s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] /
    s4_stats['s4_total_balls'] * 100
).round(2)

s4_stats['s4_score'] = (
    s4_stats['s4_avg'] * s4_stats['s4_sr'] / 100
).round(2)

# apply minimum qualifying innings
s4_stats = s4_stats[s4_stats['s4_innings'] >= min_s4_innings]

print(f"Batters with {min_s4_innings}+ qualifying innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_avg', 's4_sr', 's4_score']
].head(10))

S4 min balls per innings (median): 10
S4 min qualifying innings (75th percentile): 4
Innings with 10+ recovery balls: 997
Batters with 4+ qualifying innings: 75

Top 10 by chase recovery score:
             batter  s4_innings  s4_total_runs  s4_avg   s4_sr  s4_score
2        A Flintoff           7            245   245.0  123.74    303.16
213     LRPL Taylor          12            260   260.0   89.97    233.92
187    KIC Asalanka           6            217   217.0   89.30    193.78
43        BA Stokes           5            174   174.0  107.41    186.89
292       RG Sharma           9            220   220.0   80.29    176.64
7    AB de Villiers          14            364   182.0   96.55    175.72
161         JE Root           8            186   186.0   84.93    157.97
183      KA Pollard           5            144   144.0  109.09    157.09
105      EJG Morgan          17            369   184.5   76.40    140.96
238      MN Samuels           4            131   131.0   94.93    124.36


In [36]:
print("s4_dismissals distribution:")
print(s4_stats['s4_dismissals'].describe())

flintoff_check = s4_innings[s4_innings['batter'] == 'A Flintoff']
print(flintoff_check)

s4_dismissals distribution:
count    75.000000
mean      1.760000
std       1.344056
min       0.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       6.000000
Name: s4_dismissals, dtype: float64
        batter match_id  s4_runs  s4_balls  got_out
7   A Flintoff   247494        5         5        0
8   A Flintoff   258474        5         8        1
9   A Flintoff   361046       41        30        1
10  A Flintoff    64843       55        52        0
11  A Flintoff    64844       49        35        0
12  A Flintoff    64845        0         3        0
13  A Flintoff    65030       26        13        0
14  A Flintoff    65246        0         1        1
15  A Flintoff    66299       47        37        0
16  A Flintoff    66302        6        13        0
17  A Flintoff    66306       21        18        0


In [37]:
# corrected S4 metric - remove flawed average, use SR + runs volume

s4_stats = s4_innings.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_runs', 'sum'),
    s4_total_balls=('s4_balls', 'sum')
).reset_index()

# strike rate - the only valid metric here
s4_stats['s4_sr'] = (
    s4_stats['s4_total_runs'] / s4_stats['s4_total_balls'] * 100
).round(2)

# average runs per innings (not per dismissal - just volume per appearance)
s4_stats['s4_runs_per_innings'] = (
    s4_stats['s4_total_runs'] / s4_stats['s4_innings']
).round(2)

# combined score: SR x runs_per_innings (normalized later)
s4_stats['s4_score'] = (
    s4_stats['s4_sr'] * s4_stats['s4_runs_per_innings'] / 100
).round(2)

# apply same minimum innings threshold
s4_stats = s4_stats[s4_stats['s4_innings'] >= min_s4_innings]

print(f"Batters with {min_s4_innings}+ qualifying innings: {len(s4_stats)}")
print("\nTop 10 by chase recovery score:")
print(s4_stats.sort_values('s4_score', ascending=False)[
    ['batter', 's4_innings', 's4_total_runs',
     's4_sr', 's4_runs_per_innings', 's4_score']
].head(10))

Batters with 4+ qualifying innings: 172

Top 10 by chase recovery score:
                    batter  s4_innings  s4_total_runs   s4_sr  \
178              H Klaasen           6            143  134.91   
384  Nazmul Hossain Shanto           4             98  122.50   
3               A Flintoff          11            255  118.60   
289               L Ronchi           4             81  132.79   
67               BA Stokes           7            179  101.70   
265           KIC Asalanka           8            218   86.51   
394              PG Fulton           4            106   84.80   
260             KA Pollard           7            150  104.17   
513                TM Head           6            162   81.00   
370               N Pooran           6            135   94.41   

     s4_runs_per_innings  s4_score  
178                23.83     32.15  
384                24.50     30.01  
3                  23.18     27.49  
289                20.25     26.89  
67                 25.57  

### Component 4 (S4) — Redesigned: Chase Rescue Total Contribution

### Why redesigned
Original S4 only measured runs scored WITHIN the pressure window
(over < 25, wickets >= 3 simultaneously true). This missed the
batter's full rescue impact — many match-winning innings
accelerate AFTER the immediate pressure eases, with the bulk
of runs coming later in the innings.

### New definition
Step 1: Identify the batter's FIRST ball faced in each innings.
Step 2: Check if at that first ball, team was 3+ wickets down
        within first 10 overs of the chase (entry trigger).
Step 3: If entry trigger is met AND team won the match,
        count the batter's FULL innings total (not just the
        windowed portion).

### Why over<10 instead of over<25
Since we now measure the entire innings rather than a windowed
portion, the entry trigger should identify a sharper, more
genuine "walked into crisis" moment rather than a long stretch.
Over<10 with 3+ wickets down represents true early chase crisis.

### Metric
s4_score = Strike Rate × Runs per innings / 100
Using FULL innings runs and balls, not windowed portion.

In [38]:
# Step 1: find each batter's FIRST ball faced in each innings
first_ball = master_df[master_df['wides']==0].sort_values(
    ['match_id', 'innings', 'batter', 'over_num', 'ball_num']
).groupby(['match_id', 'innings', 'batter']).first().reset_index()

print(f"Total first-ball records: {len(first_ball)}")

# Step 2: check entry trigger - 3+ wickets down, over<10, innings 2
entry_trigger = first_ball[
    (first_ball['innings'] == 2) &
    (first_ball['cumulative_wickets'] >= 3) &
    (first_ball['over_num'] < 10)
].copy()

print(f"Batters entering during early chase crisis: {len(entry_trigger)}")

# Step 3: filter to only matches that were WON
entry_trigger = entry_trigger[
    entry_trigger['match_id'].isin(won_chases)
]

print(f"Entries that led to wins: {len(entry_trigger)}")
print(f"Unique batters: {entry_trigger['batter'].nunique()}")

Total first-ball records: 44400
Batters entering during early chase crisis: 646
Entries that led to wins: 125
Unique batters: 84


In [39]:
# get the qualifying match_id + batter + innings combinations
qualifying_entries = entry_trigger[['match_id', 'innings', 'batter']].copy()

# pull FULL innings stats for these batters in these matches
full_innings_stats = master_df[master_df['wides']==0].merge(
    qualifying_entries, on=['match_id', 'innings', 'batter']
).groupby(['batter', 'match_id']).agg(
    s4_full_runs=('runs_off_bat', 'sum'),
    s4_full_balls=('runs_off_bat', 'count')
).reset_index()

print(f"Full innings records: {len(full_innings_stats)}")
print(full_innings_stats.head(10))

# per batter aggregate
s4_stats_v2 = full_innings_stats.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_full_runs', 'sum'),
    s4_total_balls=('s4_full_balls', 'sum')
).reset_index()

s4_stats_v2['s4_sr'] = (
    s4_stats_v2['s4_total_runs'] / s4_stats_v2['s4_total_balls'] * 100
).round(2)

s4_stats_v2['s4_runs_per_innings'] = (
    s4_stats_v2['s4_total_runs'] / s4_stats_v2['s4_innings']
).round(2)

s4_stats_v2['s4_score'] = (
    s4_stats_v2['s4_sr'] * s4_stats_v2['s4_runs_per_innings'] / 100
).round(2)

print(f"\nTotal batters before threshold: {len(s4_stats_v2)}")
print("\nInnings count distribution:")
print(s4_stats_v2['s4_innings'].describe())

Full innings records: 125
           batter match_id  s4_full_runs  s4_full_balls
0      A Flintoff   361046            41             30
1      A Flintoff    66299            47             37
2       A McGrath    66299             1              5
3       A Symonds   249229            38             40
4       A Symonds    65653            73             57
5  AB de Villiers   520600           106            106
6  AB de Villiers   534232            75             79
7  AB de Villiers   800477           101             97
8        AR Adams    65270            36             20
9        AT Carey  1317479            85             99

Total batters before threshold: 84

Innings count distribution:
count    84.000000
mean      1.488095
std       0.799006
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       4.000000
Name: s4_innings, dtype: float64


In [40]:
# loosen both: wickets>=2 AND over<20
entry_trigger_v3 = first_ball[
    (first_ball['innings'] == 2) &
    (first_ball['cumulative_wickets'] >= 2) &
    (first_ball['over_num'] < 10)
].copy()

entry_trigger_v3 = entry_trigger_v3[
    entry_trigger_v3['match_id'].isin(won_chases)
]

print(f"Entries with wickets>=2, over<10: {len(entry_trigger_v3)}")
print(f"Unique batters: {entry_trigger_v3['batter'].nunique()}")
# loosen both: wickets>=2 AND over<20
entry_trigger_v3 = first_ball[
    (first_ball['innings'] == 2) &
    (first_ball['cumulative_wickets'] >= 2) &
    (first_ball['over_num'] < 20)
].copy()

entry_trigger_v3 = entry_trigger_v3[
    entry_trigger_v3['match_id'].isin(won_chases)
]

print(f"Entries with wickets>=2, over<20: {len(entry_trigger_v3)}")
print(f"Unique batters: {entry_trigger_v3['batter'].nunique()}")

Entries with wickets>=2, over<10: 499
Unique batters: 208
Entries with wickets>=2, over<20: 1389
Unique batters: 383


In [41]:
qualifying_entries_final = entry_trigger[
    (entry_trigger['cumulative_wickets'] >= 2) &
    (entry_trigger['over_num'] < 10)
][['match_id', 'innings', 'batter']].copy()

# wait - need to rebuild entry_trigger with correct filters first
entry_trigger_final = first_ball[
    (first_ball['innings'] == 2) &
    (first_ball['cumulative_wickets'] >= 2) &
    (first_ball['over_num'] < 10)
].copy()

entry_trigger_final = entry_trigger_final[
    entry_trigger_final['match_id'].isin(won_chases)
]

qualifying_entries_final = entry_trigger_final[['match_id', 'innings', 'batter']].copy()

full_innings_final = master_df[master_df['wides']==0].merge(
    qualifying_entries_final, on=['match_id', 'innings', 'batter']
).groupby(['batter', 'match_id']).agg(
    s4_full_runs=('runs_off_bat', 'sum'),
    s4_full_balls=('runs_off_bat', 'count')
).reset_index()

s4_stats_final = full_innings_final.groupby('batter').agg(
    s4_innings=('match_id', 'count'),
    s4_total_runs=('s4_full_runs', 'sum'),
    s4_total_balls=('s4_full_balls', 'sum')
).reset_index()

print("Innings count distribution:")
print(s4_stats_final['s4_innings'].describe())
print(f"\n75th percentile: {s4_stats_final['s4_innings'].quantile(0.75)}")

Innings count distribution:
count    208.000000
mean       2.399038
std        2.469016
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       17.000000
Name: s4_innings, dtype: float64

75th percentile: 2.0


### S4 Design Exploration — Full Innings Approach (Tested, Not Adopted)

### Idea explored
Redesign S4 to identify batters who walked in during early chase
crisis (first ball faced while 2-3+ wickets down, early overs),
then measure their FULL innings contribution rather than just
runs scored within a pressure window.

### Why this is conceptually strong
This captures the complete rescue act — many match-winning innings
accelerate AFTER immediate pressure eases, with bulk of runs coming
later. A windowed metric misses this full picture.

### Why not adopted — statistical insufficiency
Testing multiple threshold combinations:
| Wickets | Over window | Qualifying entries | Unique batters | 75th pct innings |
|---|---|---|---|---|
| >=3 | <10 | 125 | 84 | ~2 |
| >=2 | <10 | 499 | 208 | 2.0 |
| >=2 | <20 | 1389 | 383 | not tested further |

The triple constraint (entry trigger + won chases only + first-ball
match) made every reasonable threshold combination too sparse for
reliable ranking — even top qualifying batters had only 1-2 
qualifying innings, insufficient for statistical confidence.

### Decision
Reverted to windowed S4 approach (over<25, wickets>=3, SR x 
runs_per_innings metric) which had been validated with 172 
qualified batters and cricket-sensible Top 10 (Klaasen, Flintoff,
Stokes, Pollard).

### Future improvement opportunity
The full-innings approach remains conceptually valuable. With a
larger dataset (more historical matches, or including near-misses
not just won chases), this could become statistically viable.
Noted as a future enhancement for when more data becomes available.

In [42]:
print("Final S4 (Path C - windowed approach):")
print(f"Qualified batters: {len(s4_stats)}")
print(s4_stats[['batter', 's4_innings', 's4_sr', 
                 's4_runs_per_innings', 's4_score']].sort_values('s4_score', ascending=False).head(10))

Final S4 (Path C - windowed approach):
Qualified batters: 172
                    batter  s4_innings   s4_sr  s4_runs_per_innings  s4_score
178              H Klaasen           6  134.91                23.83     32.15
384  Nazmul Hossain Shanto           4  122.50                24.50     30.01
3               A Flintoff          11  118.60                23.18     27.49
289               L Ronchi           4  132.79                20.25     26.89
67               BA Stokes           7  101.70                25.57     26.00
265           KIC Asalanka           8   86.51                27.25     23.57
394              PG Fulton           4   84.80                26.50     22.47
260             KA Pollard           7  104.17                21.43     22.32
513                TM Head           6   81.00                27.00     21.87
370               N Pooran           6   94.41                22.50     21.24


# PPI — Pressure Performance Index

## Overview
PPI measures how batters perform specifically under pressure situations.
Traditional cricket ratings reward volume — career averages and total runs.
PPI rewards value — runs scored when they matter most.

A batter averaging 45 in comfortable situations is fundamentally different
from one averaging 45 when team is 3 down, chasing 280, in a World Cup knockout.
PPI captures this difference. Traditional ratings do not.

## Why PPI alone is not the final ranking
PPI is one of three components in the final player score:
- PPI (35%) → pressure performance
- CFS (35%) → clutch factor, match-winning contributions
- ENS (30%) → era-adjusted overall batting quality

PPI alone would over-reward pressure specialists who lack overall quality.
Combined with CFS and ENS, it forms a complete picture of batting greatness.

## Architecture — 4 pressure situations

| Situation | Description | Metric | Weight |
|---|---|---|---|
| S1 | Early collapse, innings 1 | Strike Rate | 25% |
| S2 | Death chase, overs 40-50 | SR × Win% | 30% |
| S3 | World Cup performance | Avg + Consistency | 25% |
| S4 | Chase recovery, early collapse won | Avg × SR | 20% |

## Minimum career balls threshold

### Problem with arbitrary thresholds
Original threshold = 500 balls (arbitrary, not justified).
A player with 499 balls excluded, one with 501 included —
no statistical reason for this cutoff.

### Statistical derivation
Distribution of total career balls across all 1,897 batters:
| Statistic | Value |
|---|---|
| Count | 1,897 batters |
| Mean | 694.94 balls |
| Median | 169.00 balls |
| 25th percentile | 41.00 balls |
| 75th percentile | 584.00 balls |
| Max | 15,652 balls |

Threshold = 75th percentile = 584 balls minimum.

### Why 75th percentile
We are ranking Top 100 ODI batters of all time.
This requires genuine established ODI batters, not occasional players.
Removes batters with tiny sample sizes.

## Missing data handling
Batters absent from a situation receive population mean score.
Not zero — zero would unfairly punish batters who never
played in that specific situation.

Example:
- A top order batter rarely faces S4 (chase recovery)
- Not because they are bad — just never needed to rescue
- Population mean gives them neutral score, not penalty

## Statistical threshold summary

| Situation | Threshold Type | Method | Value |
|---|---|---|---|
| S1 | Min career pressure balls | Mean | 60 balls |
| S2 | Min balls per innings | Median | 9 balls |
| S2 | Min qualifying innings | 75th percentile | 5 innings |
| S3 | Min World Cup innings | 75th percentile + 1 | 9 innings |
| S4 | Min balls per innings | Median | 10 balls |
| S4 | Min qualifying innings | 75th percentile | 4 innings |
| PPI | Min career total balls | 75th percentile | 584 balls |

All thresholds derived from actual data distributions.
No arbitrary values used anywhere in the pipeline.

## Weights justification

| Situation | Weight | Reason |
|---|---|---|
| S1 (25%) | Early collapse | Common situation, large sample, first innings |
| S2 (30%) | Death chase | Highest match-winning impact, most visible pressure |
| S3 (25%) | World Cup | Tournament defines legacies, rare opportunity |
| S4 (20%) | Chase recovery | Rarest skill, smaller sample size |

S2 weighted highest because death chase performance
is the most visible and impactful pressure situation in ODI cricket.
Matches are won or lost in overs 40-50 more than any other phase.
S4 weighted lowest because sample size is inherently smaller —
early collapse wins are rarer than other situations.

## Final PPI formula

PPI = (0.25 × s1_norm) +

(0.30 × s2_norm) +
(0.20 × s3_norm) +

(0.05 × s3_con_norm) +

(0.20 × s4_norm)

Note: S3 contributes 25% total split as:
- s3_norm (average) = 20%
- s3_con_norm (consistency) = 5%

## Data scope limitation
Ball-by-ball data available from 2002 onwards only.
Pre-2002 legends (Sachin 1989-2001, Lara 1990-2001) are
underrepresented in pressure situation samples.
This is partially corrected by ENS (Era Normalisation Score)
which adjusts for career span and era scoring conditions.
Pre-2002 performance captured through career averages in ENS component.



In [43]:
from sklearn.preprocessing import MinMaxScaler
# statistically derived career balls threshold

total_balls = master_df[
    master_df['wides'] == 0
].groupby('batter')['runs_off_bat'].count().reset_index()
total_balls.columns = ['batter', 'total_balls']

# statistically derived career balls threshold
min_career_balls = int(total_balls['total_balls'].quantile(0.75))
print(f"Minimum career balls (75th percentile): {min_career_balls}")

qualified = total_balls[total_balls['total_balls'] >= min_career_balls]
print(f"Qualified batters: {len(qualified)}")

# start PPI dataframe with qualified batters
ppi_df = qualified.copy()

# merge all 4 situations
ppi_df = ppi_df.merge(
    s1_stats[['batter', 's1_sr']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s2_stats[['batter', 's2_score']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s3_stats[['batter', 's3_avg', 's3_consistency']], 
    on='batter', how='left'
)
ppi_df = ppi_df.merge(
    s4_stats[['batter', 's4_score']], 
    on='batter', how='left'
)

print(f"\nBefore filling nulls:")
print(ppi_df.isnull().sum())

# fill missing with population mean
ppi_df['s1_sr'] = ppi_df['s1_sr'].fillna(ppi_df['s1_sr'].mean())
ppi_df['s2_score'] = ppi_df['s2_score'].fillna(ppi_df['s2_score'].mean())
ppi_df['s3_avg'] = ppi_df['s3_avg'].fillna(ppi_df['s3_avg'].mean())
ppi_df['s3_consistency'] = ppi_df['s3_consistency'].fillna(
    ppi_df['s3_consistency'].mean()
)
ppi_df['s4_score'] = ppi_df['s4_score'].fillna(ppi_df['s4_score'].mean())

print(f"\nAfter filling nulls:")
print(ppi_df.isnull().sum())




Minimum career balls (75th percentile): 584
Qualified batters: 475

Before filling nulls:
batter              0
total_balls         0
s1_sr             243
s2_score          383
s3_avg            314
s3_consistency    314
s4_score          303
dtype: int64

After filling nulls:
batter            0
total_balls       0
s1_sr             0
s2_score          0
s3_avg            0
s3_consistency    0
s4_score          0
dtype: int64


In [44]:
# statistical analysis of total career balls
print("Total career balls per batter distribution:")
print(total_balls['total_balls'].describe())
print(f"Mean:            {total_balls['total_balls'].mean():.2f}")
print(f"Median:          {total_balls['total_balls'].median():.2f}")
print(f"25th percentile: {total_balls['total_balls'].quantile(0.25):.2f}")
print(f"75th percentile: {total_balls['total_balls'].quantile(0.75):.2f}")

Total career balls per batter distribution:
count     1897.000000
mean       694.938851
std       1480.347163
min          1.000000
25%         41.000000
50%        169.000000
75%        584.000000
max      15652.000000
Name: total_balls, dtype: float64
Mean:            694.94
Median:          169.00
25th percentile: 41.00
75th percentile: 584.00


In [45]:
# normalise each component to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))

ppi_df['s1_norm'] = scaler.fit_transform(ppi_df[['s1_sr']])
ppi_df['s2_norm'] = scaler.fit_transform(ppi_df[['s2_score']])
ppi_df['s3_norm'] = scaler.fit_transform(ppi_df[['s3_avg']])
ppi_df['s3_con_norm'] = scaler.fit_transform(ppi_df[['s3_consistency']])
ppi_df['s4_norm'] = scaler.fit_transform(ppi_df[['s4_score']])

# weighted PPI
ppi_df['PPI'] = (
    0.25 * ppi_df['s1_norm'] +
    0.30 * ppi_df['s2_norm'] +
    0.20 * ppi_df['s3_norm'] +
    0.05 * ppi_df['s3_con_norm'] +
    0.20 * ppi_df['s4_norm']
).round(2)

# sort by PPI
ppi_df = ppi_df.sort_values('PPI', ascending=False).reset_index(drop=True)
ppi_df['PPI_rank'] = ppi_df.index + 1

print(f"\nTotal qualified batters: {len(ppi_df)}")
print("\nTop 20 by PPI:")
print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's4_norm',
              'total_balls']].head(20))


Total qualified batters: 475

Top 20 by PPI:
    PPI_rank                 batter    PPI     s1_norm     s2_norm  \
0          1                V Kohli  63.82   40.909835   93.118995   
1          2              BA Stokes  63.45   33.775160   71.574279   
2          3            CJ Anderson  58.12  100.000000   41.078200   
3          4                JE Root  57.39   43.937162   87.450111   
4          5               L Ronchi  56.85   27.778095   75.898004   
5          6               KL Rahul  56.46   29.790542   80.576497   
6          7         AB de Villiers  56.35   31.729668   51.825573   
7          8               SK Raina  55.68   30.682376   89.733925   
8          9              A Symonds  54.23   35.076092   41.078200   
9         10  Nazmul Hossain Shanto  53.94   50.155457   41.078200   
10        11          KS Williamson  53.56   24.136802   77.228381   
11        12              H Klaasen  52.77   29.144166   41.078200   
12        13             KJ O'Brien  51.96  

In [46]:
from sklearn.preprocessing import MinMaxScaler

# rebuild PPI combination with corrected S4
ppi_df = qualified.copy()

ppi_df = ppi_df.merge(s1_stats[['batter', 's1_sr']], on='batter', how='left')
ppi_df = ppi_df.merge(s2_stats[['batter', 's2_score']], on='batter', how='left')
ppi_df = ppi_df.merge(s3_stats[['batter', 's3_avg', 's3_consistency']], on='batter', how='left')
ppi_df = ppi_df.merge(s4_stats[['batter', 's4_score']], on='batter', how='left')

# fill missing with population mean
ppi_df['s1_sr'] = ppi_df['s1_sr'].fillna(ppi_df['s1_sr'].mean())
ppi_df['s2_score'] = ppi_df['s2_score'].fillna(ppi_df['s2_score'].mean())
ppi_df['s3_avg'] = ppi_df['s3_avg'].fillna(ppi_df['s3_avg'].mean())
ppi_df['s3_consistency'] = ppi_df['s3_consistency'].fillna(ppi_df['s3_consistency'].mean())
ppi_df['s4_score'] = ppi_df['s4_score'].fillna(ppi_df['s4_score'].mean())

# normalise
scaler = MinMaxScaler(feature_range=(0, 100))
ppi_df['s1_norm'] = scaler.fit_transform(ppi_df[['s1_sr']])
ppi_df['s2_norm'] = scaler.fit_transform(ppi_df[['s2_score']])
ppi_df['s3_norm'] = scaler.fit_transform(ppi_df[['s3_avg']])
ppi_df['s3_con_norm'] = scaler.fit_transform(ppi_df[['s3_consistency']])
ppi_df['s4_norm'] = scaler.fit_transform(ppi_df[['s4_score']])

# weighted PPI
ppi_df['PPI'] = (
    0.25 * ppi_df['s1_norm'] +
    0.30 * ppi_df['s2_norm'] +
    0.15 * ppi_df['s3_norm'] +
    0.10 * ppi_df['s3_con_norm'] +
    0.20 * ppi_df['s4_norm']
).round(2)

ppi_df = ppi_df.sort_values('PPI', ascending=False).reset_index(drop=True)
ppi_df['PPI_rank'] = ppi_df.index + 1

print(f"Total qualified batters: {len(ppi_df)}")
print("\nTop 20 by PPI (with corrected S4):")
print(ppi_df[['PPI_rank', 'batter', 'PPI', 's1_norm', 's2_norm', 
              's3_norm', 's4_norm']].head(20))

Total qualified batters: 475

Top 20 by PPI (with corrected S4):
    PPI_rank                 batter    PPI     s1_norm     s2_norm  \
0          1                V Kohli  64.66   40.909835   93.118995   
1          2              BA Stokes  63.68   33.775160   71.574279   
2          3            CJ Anderson  58.56  100.000000   41.078200   
3          4                JE Root  57.98   43.937162   87.450111   
4          5               L Ronchi  57.29   27.778095   75.898004   
5          6               KL Rahul  56.61   29.790542   80.576497   
6          7         AB de Villiers  56.08   31.729668   51.825573   
7          8               SK Raina  55.86   30.682376   89.733925   
8          9  Nazmul Hossain Shanto  54.38   50.155457   41.078200   
9         10          KS Williamson  54.32   24.136802   77.228381   
10        11              H Klaasen  53.09   29.144166   41.078200   
11        12              RG Sharma  52.58   27.745050   64.552846   
12        13             

In [47]:
print(ppi_df.columns.tolist())

['batter', 'total_balls', 's1_sr', 's2_score', 's3_avg', 's3_consistency', 's4_score', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'PPI', 'PPI_rank']


In [48]:
kohli = ppi_df[ppi_df['batter'] == 'V Kohli']
print(kohli[['batter', 'PPI', 's1_norm', 's2_norm', 
             's3_norm', 's3_con_norm', 's4_norm']].to_string())

    batter    PPI    s1_norm    s2_norm    s3_norm  s3_con_norm    s4_norm
0  V Kohli  64.66  40.909835  93.118995  68.650078    85.426417  38.289269


In [49]:
# show all columns properly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(ppi_df[['PPI_rank', 'batter', 'PPI', 
              's1_norm', 's2_norm', 
              's3_norm', 's3_con_norm',
              's4_norm',
              'total_balls']].head(20).to_string())

    PPI_rank                 batter    PPI     s1_norm     s2_norm     s3_norm  s3_con_norm     s4_norm  total_balls
0          1                V Kohli  64.66   40.909835   93.118995   68.650078    85.426417   38.289269        15652
1          2              BA Stokes  63.68   33.775160   71.574279   68.493809    73.127925   80.870918         3616
2          3            CJ Anderson  58.56  100.000000   41.078200   39.918349    48.579148   51.944012         1012
3          4                JE Root  57.98   43.937162   87.450111   50.342589    62.181487   34.961120         8415
4          5               L Ronchi  57.29   27.778095   75.898004   39.918349    48.579148   83.639191         1220
5          6               KL Rahul  56.61   29.790542   80.576497   69.227071    72.230889   36.889580         3596
6          7         AB de Villiers  56.08   31.729668   51.825573   82.173338    76.820073   62.954899         9323
7          8               SK Raina  55.86   30.682376   89.7339

In [50]:
ppi_df.head(1)

,batter,total_balls,s1_sr,s2_score,s3_avg,s3_consistency,s4_score,s1_norm,s2_norm,s3_norm,s3_con_norm,s4_norm,PPI,PPI_rank
0,V Kohli,15652,75.0,125.99,59.75,65.71,12.31,40.909835,93.118995,68.650078,85.426417,38.289269,64.66,1


In [51]:
ppi_df.to_csv(
    r"D:\CricMetric-AI\data\processed\ppi_scores.csv",
    index=False
)
print(f"Saved. Total batters: {len(ppi_df)}")

Saved. Total batters: 475


In [52]:
print(master_df.shape)
print(ppi_df.shape)

(1349408, 30)
(475, 14)


In [53]:
print(ppi_df.columns.tolist())
print(ppi_df.head())

['batter', 'total_balls', 's1_sr', 's2_score', 's3_avg', 's3_consistency', 's4_score', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'PPI', 'PPI_rank']
        batter  total_balls       s1_sr    s2_score     s3_avg  \
0      V Kohli        15652   75.000000  125.990000  59.750000   
1    BA Stokes         3616   66.280000   96.840000  59.620000   
2  CJ Anderson         1012  147.220000   55.578804  35.848075   
3      JE Root         8415   78.700000  118.320000  44.520000   
4     L Ronchi         1220   58.950388  102.690000  35.848075   

   s3_consistency  s4_score     s1_norm    s2_norm    s3_norm  s3_con_norm  \
0       65.710000     12.31   40.909835  93.118995  68.650078    85.426417   
1       56.250000     26.00   33.775160  71.574279  68.493809    73.127925   
2       37.367081     16.70  100.000000  41.078200  39.918349    48.579148   
3       47.830000     11.24   43.937162  87.450111  50.342589    62.181487   
4       37.367081     26.89   27.778095  75.8980

# CFS — Clutch Factor Score

## Overview
CFS measures match-winning contributions — not just runs scored,
but runs that actually won matches.

A batter who scores 20 fifties in winning causes is fundamentally
more valuable than one who scores 20 fifties in losing causes.
PPI measures pressure performance; CFS measures match-winning impact.
Together with ENS, these three form the complete player evaluation.

## CFS Components
1. Win% when player scored 50+ (vs baseline team win%)
2. Match-winning innings ratio (top scorer in a win)
3. Big score conversion (50+ scores that led to wins)

In [54]:
team_totals = master_df[master_df['wides']==0].groupby(
    ['match_id', 'innings', 'batting_team']
)['total_runs_ball'].sum().reset_index()
team_totals.columns = ['match_id', 'innings', 'batting_team', 'team_total_runs']

print(team_totals.head(10))
print(f"\nTotal team-innings records: {len(team_totals)}")

  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              259
1  1000887        2     Pakistan              171
2  1000889        1    Australia              209
3  1000889        2     Pakistan              214
4  1000891        1     Pakistan              255
5  1000891        2    Australia              260
6  1000893        1    Australia              353
7  1000893        2     Pakistan              263
8  1000895        1    Australia              363
9  1000895        2     Pakistan              299

Total team-innings records: 5033


In [55]:
team_totals = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['cumulative_runs'].max().reset_index()
team_totals.columns = ['match_id', 'innings', 'batting_team', 'team_total_runs']

print(team_totals.head(10))
print(f"\nTotal team-innings records: {len(team_totals)}")

  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              268
1  1000887        2     Pakistan              176
2  1000889        1    Australia              220
3  1000889        2     Pakistan              221
4  1000891        1     Pakistan              263
5  1000891        2    Australia              265
6  1000893        1    Australia              353
7  1000893        2     Pakistan              267
8  1000895        1    Australia              369
9  1000895        2     Pakistan              312

Total team-innings records: 5033


In [56]:
# check against actual cricket scorecards - Australia vs Pakistan, 2017
# this was the Brisbane match from our very first sample file
print("Match 1000887 (Australia vs Pakistan, Brisbane 2017):")
print(team_totals[team_totals['match_id']=='1000887'])

Match 1000887 (Australia vs Pakistan, Brisbane 2017):
  match_id  innings batting_team  team_total_runs
0  1000887        1    Australia              268
1  1000887        2     Pakistan              176


In [57]:
# Total 1: real match score (includes everything - for win/loss)
match_score = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['cumulative_runs'].max().reset_index()
match_score.columns = ['match_id', 'innings', 'batting_team', 'team_match_total']

# Total 2: pure batting total (runs_off_bat only - for contribution share)
# no need to filter wides since runs_off_bat is already pure
batting_total = master_df.groupby(
    ['match_id', 'innings', 'batting_team']
)['runs_off_bat'].sum().reset_index()
batting_total.columns = ['match_id', 'innings', 'batting_team', 'team_batting_total']

print("Match score (real total, includes extras):")
print(match_score.head())
print("\nBatting total (pure runs_off_bat, no extras):")
print(batting_total.head())

Match score (real total, includes extras):
  match_id  innings batting_team  team_match_total
0  1000887        1    Australia               268
1  1000887        2     Pakistan               176
2  1000889        1    Australia               220
3  1000889        2     Pakistan               221
4  1000891        1     Pakistan               263

Batting total (pure runs_off_bat, no extras):
  match_id  innings batting_team  team_batting_total
0  1000887        1    Australia                 257
1  1000887        2     Pakistan                 162
2  1000889        1    Australia                 202
3  1000889        2     Pakistan                 208
4  1000891        1     Pakistan                 248


In [58]:
# filter to only innings 1 and 2 for main match result
match_score_main = match_score[match_score['innings'].isin([1, 2])]

match_pivot = match_score_main.pivot(
    index='match_id', columns='innings', values='team_match_total'
).reset_index()
match_pivot.columns = ['match_id', 'innings1_total', 'innings2_total']

team_names = match_score_main.pivot(
    index='match_id', columns='innings', values='batting_team'
).reset_index()
team_names.columns = ['match_id', 'team_innings1', 'team_innings2']

match_results = match_pivot.merge(team_names, on='match_id')

match_results['winner'] = match_results.apply(
    lambda row: row['team_innings1'] if row['innings1_total'] > row['innings2_total']
    else row['team_innings2'] if row['innings2_total'] > row['innings1_total']
    else 'Tie',
    axis=1
)

print(match_results.head(10))
print(f"\nTotal matches with results: {len(match_results)}")
print(f"\nResult distribution:")
print(match_results['winner'].value_counts().head())

  match_id  innings1_total  innings2_total team_innings1 team_innings2  \
0  1000887           268.0           176.0     Australia      Pakistan   
1  1000889           220.0           221.0     Australia      Pakistan   
2  1000891           263.0           265.0      Pakistan     Australia   
3  1000893           353.0           267.0     Australia      Pakistan   
4  1000895           369.0           312.0     Australia      Pakistan   
5  1001371           324.0           256.0     Australia   New Zealand   
6  1001373           378.0           262.0     Australia   New Zealand   
7  1001375           264.0           147.0     Australia   New Zealand   
8  1004283           153.0           136.0      Scotland     Hong Kong   
9  1004285           266.0           213.0      Scotland     Hong Kong   

      winner  
0  Australia  
1   Pakistan  
2  Australia  
3  Australia  
4  Australia  
5  Australia  
6  Australia  
7  Australia  
8   Scotland  
9   Scotland  

Total matches with 

In [59]:
# get each player's runs per innings + their team's batting total
player_innings = master_df.groupby(
    ['batter', 'match_id', 'innings', 'batting_team']
).agg(
    player_runs=('runs_off_bat', 'sum'),
    player_balls=('runs_off_bat', 'count'),
    got_out=('is_wicket', 'max')
).reset_index()

# merge team batting total
player_innings = player_innings.merge(
    batting_total, on=['match_id', 'innings', 'batting_team']
)

# compute contribution share
player_innings['contribution_share'] = (
    player_innings['player_runs'] / player_innings['team_batting_total']
).round(4)

# merge match winner info
player_innings = player_innings.merge(
    match_results[['match_id', 'winner']], on='match_id', how='left'
)

# did this player's team win?
player_innings['team_won'] = (
    player_innings['batting_team'] == player_innings['winner']
).astype(int)

print(player_innings.head(10))
print(f"\nTotal player-innings records: {len(player_innings)}")

       batter match_id  innings batting_team  player_runs  player_balls  \
0     A Ashok  1388212        1  New Zealand           10            12   
1  A Athanaze  1373576        2  West Indies           66            65   
2  A Athanaze  1373577        1  West Indies            4            14   
3  A Athanaze  1373578        2  West Indies           45            51   
4  A Athanaze  1375847        1  West Indies            5            11   
5  A Athanaze  1375848        2  West Indies           11            14   
6  A Athanaze  1375849        1  West Indies           32            64   
7  A Athanaze  1377007        2  West Indies           65            48   
8  A Athanaze  1381214        1  West Indies           22            19   
9  A Athanaze  1381215        2  West Indies            6             9   

   got_out  team_batting_total  contribution_share       winner  team_won  
0        1                  94              0.1064   Bangladesh         0  
1        1            

In [60]:
# filter to only 50+ scores (the "big innings" we care about for clutch)
big_scores = player_innings[player_innings['player_runs'] >= 50].copy()

print(f"Total 50+ innings: {len(big_scores)}")
print(f"Unique batters with 50+: {big_scores['batter'].nunique()}")

# simple win% when scored 50+
simple_cfs = big_scores.groupby('batter').agg(
    fifties_plus=('match_id', 'count'),
    wins_with_50plus=('team_won', 'sum')
).reset_index()

simple_cfs['win_pct_50plus'] = (
    simple_cfs['wins_with_50plus'] / simple_cfs['fifties_plus'] * 100
).round(2)

print("\nTop 10 by win% when scoring 50+:")
print(simple_cfs.sort_values('win_pct_50plus', ascending=False).head(10))

Total 50+ innings: 7040
Unique batters with 50+: 792

Top 10 by win% when scoring 50+:
                batter  fifties_plus  wins_with_50plus  win_pct_50plus
775          Wasim Ali             1                 1           100.0
10            A Sharma             5                 5           100.0
791       Zubayr Hamza             1                 1           100.0
0           A Athanaze             2                 2           100.0
774  Washington Sundar             1                 1           100.0
768        WS Jayantha             1                 1           100.0
779        YBK Jaiswal             1                 1           100.0
314        JD Campbell             1                 1           100.0
312      JAD Lawrenson             1                 1           100.0
343         JP Greaves             1                 1           100.0


In [61]:
print("Fifties_plus distribution:")
print(simple_cfs['fifties_plus'].describe())
print(f"\nMean: {simple_cfs['fifties_plus'].mean():.2f}")
print(f"Median: {simple_cfs['fifties_plus'].median():.2f}")
print(f"75th percentile: {simple_cfs['fifties_plus'].quantile(0.75):.2f}")

Fifties_plus distribution:
count    792.000000
mean       8.888889
std       14.075569
min        1.000000
25%        1.000000
50%        3.000000
75%        9.250000
max      129.000000
Name: fifties_plus, dtype: float64

Mean: 8.89
Median: 3.00
75th percentile: 9.25


In [62]:
min_fifties = int(simple_cfs['fifties_plus'].quantile(0.75)) + 1
print(f"Minimum 50+ scores threshold (75th percentile): {min_fifties}")

simple_cfs_filtered = simple_cfs[simple_cfs['fifties_plus'] >= min_fifties]

print(f"Qualified batters: {len(simple_cfs_filtered)}")
print("\nTop 10 by win% when scoring 50+:")
print(simple_cfs_filtered.sort_values('win_pct_50plus', ascending=False).head(10))

Minimum 50+ scores threshold (75th percentile): 10
Qualified batters: 198

Top 10 by win% when scoring 50+:
           batter  fifties_plus  wins_with_50plus  win_pct_50plus
11      A Symonds            29                28           96.55
81     Aqib Ilyas            11                10           90.91
196     DR Martyn            21                19           90.48
605    RR Rossouw            10                 9           90.00
747    UT Khawaja            14                12           85.71
518      NJ Astle            13                11           84.62
706  Shubman Gill            25                21           84.00
267   HM Nicholls            18                15           83.33
341     JP Duminy            30                25           83.33
783  Yasir Hameed            12                10           83.33


### Component 4: Dominance Score
Measures how often a batter's individual contribution
crossed a scaling threshold based on team total size.

| Team Total Range | Threshold | Reasoning |
|---|---|---|
| < 100 | 25% | Even moderate share matters in small totals |
| 100-200 | 30% | Standard collapse/defensive total |
| 200-300 | 35% | Competitive total, harder to dominate |
| 300+ | 40% | Big total, true dominance needs bigger share |

dominance_count = number of innings where contribution_share
                  crossed the applicable threshold

dominance_rate = dominance_count / total_innings

In [63]:
# Component 4: Dominance Score
# scaling threshold based on team total

def get_dominance_threshold(team_total):
    if team_total < 100:
        return 0.25
    elif team_total < 200:
        return 0.30
    elif team_total < 300:
        return 0.35
    else:
        return 0.40

player_innings['dominance_threshold'] = player_innings['team_batting_total'].apply(
    get_dominance_threshold
)

player_innings['is_dominant'] = (
    player_innings['contribution_share'] >= player_innings['dominance_threshold']
).astype(int)

print(f"Total dominant innings: {player_innings['is_dominant'].sum()}")
print(f"Percentage of all innings: {player_innings['is_dominant'].mean()*100:.2f}%")

# per batter dominance stats
dominance_stats = player_innings.groupby('batter').agg(
    total_innings=('match_id', 'count'),
    dominant_innings=('is_dominant', 'sum')
).reset_index()

dominance_stats['dominance_rate'] = (
    dominance_stats['dominant_innings'] / dominance_stats['total_innings'] * 100
).round(2)

print("\nTop 10 by dominance rate:")
print(dominance_stats.sort_values('dominance_rate', ascending=False).head(10))

Total dominant innings: 3309
Percentage of all innings: 7.45%

Top 10 by dominance rate:
            batter  total_innings  dominant_innings  dominance_rate
511       FY Fazal              1                 1           100.0
376     D Daesrath              1                 1           100.0
311     CB Lambert              1                 1           100.0
263      BT Foakes              1                 1           100.0
98        AM Tribe              4                 2            50.0
695    J Figy John              2                 1            50.0
746       JF Smith              2                 1            50.0
945    L Steenkamp              4                 2            50.0
1541    SJ Massiah              2                 1            50.0
280   C Carmichael              2                 1            50.0


In [64]:
print("Total innings distribution (for dominance):")
print(dominance_stats['total_innings'].describe())
print(f"\n75th percentile: {dominance_stats['total_innings'].quantile(0.75):.2f}")

Total innings distribution (for dominance):
count    1897.000000
mean       23.410121
std        37.110501
min         1.000000
25%         3.000000
50%         9.000000
75%        27.000000
max       296.000000
Name: total_innings, dtype: float64

75th percentile: 27.00


In [65]:
min_innings_dominance = int(dominance_stats['total_innings'].quantile(0.90))
print(f"Minimum innings threshold (90th percentile): {min_innings_dominance}")

dominance_filtered = dominance_stats[
    dominance_stats['total_innings'] >= min_innings_dominance
]

print(f"Qualified batters: {len(dominance_filtered)}")
print("\nTop 10 by dominance rate:")
print(dominance_filtered.sort_values('dominance_rate', ascending=False).head(10))

Minimum innings threshold (90th percentile): 63
Qualified batters: 192

Top 10 by dominance rate:
             batter  total_innings  dominant_innings  dominance_rate
1584   SR Tendulkar            144                34           23.61
601       HG Munsey             67                15           22.39
1800        V Kohli            296                66           22.30
665       IJL Trott             65                14           21.54
876      KJ Coetzer             63                13           20.63
1600  ST Jayasuriya            133                27           20.30
99          AN Cook             91                18           19.78
604        HH Gibbs            102                20           19.61
613         HM Amla            174                34           19.54
1764    Tamim Iqbal            215                42           19.53


In [66]:
# find top scorer for each team in each match
top_scorers = player_innings.loc[
    player_innings.groupby(['match_id', 'innings', 'batting_team'])['player_runs'].idxmax()
]

print(f"Total top-scorer records: {len(top_scorers)}")
print(top_scorers[['batter', 'match_id', 'player_runs', 'team_won']].head(10))

# was this top scorer's team also the winner?
top_scorers['was_match_winning_topscore'] = (
    (top_scorers['team_won'] == 1)
).astype(int)

# per batter: how many times were they top scorer in a win
match_winning_stats = top_scorers.groupby('batter').agg(
    times_top_scorer=('match_id', 'count'),
    times_topscore_in_win=('was_match_winning_topscore', 'sum')
).reset_index()

match_winning_stats['match_winning_ratio'] = (
    match_winning_stats['times_topscore_in_win'] / 
    match_winning_stats['times_top_scorer'] * 100
).round(2)

print("\nTop scorer distribution:")
print(match_winning_stats['times_top_scorer'].describe())

Total top-scorer records: 5033
                batter match_id  player_runs  team_won
26139          MS Wade  1000887          100         1
5991        Babar Azam  1000887           33         0
36530        SPD Smith  1000889           60         0
27156  Mohammad Hafeez  1000889           72         1
5993        Babar Azam  1000891           84         0
36531        SPD Smith  1000891          108         1
8723         DA Warner  1000893          130         1
38640    Sharjeel Khan  1000893           74         0
8724         DA Warner  1000895          179         1
5995        Babar Azam  1000895          100         0

Top scorer distribution:
count    775.000000
mean       6.494194
std        9.454209
min        1.000000
25%        1.000000
50%        3.000000
75%        7.000000
max       79.000000
Name: times_top_scorer, dtype: float64


In [67]:
print(f"90th percentile: {match_winning_stats['times_top_scorer'].quantile(0.90):.2f}")

min_top_scorer = int(match_winning_stats['times_top_scorer'].quantile(0.90))
print(f"Minimum threshold: {min_top_scorer}")

match_winning_filtered = match_winning_stats[
    match_winning_stats['times_top_scorer'] >= min_top_scorer
]

print(f"Qualified batters: {len(match_winning_filtered)}")
print("\nTop 10 by match-winning ratio:")
print(match_winning_filtered.sort_values('match_winning_ratio', ascending=False).head(10))

90th percentile: 18.00
Minimum threshold: 18
Qualified batters: 79

Top 10 by match-winning ratio:
             batter  times_top_scorer  times_topscore_in_win  \
9         A Symonds                22                     22   
590      RT Ponting                33                     28   
19     AC Gilchrist                27                     21   
436       MJ Clarke                36                     26   
17   AB de Villiers                50                     36   
438      MJ Guptill                39                     28   
715      TM Dilshan                48                     34   
766     Younis Khan                19                     13   
265         HM Amla                38                     26   
734         V Kohli                79                     54   

     match_winning_ratio  
9                 100.00  
590                84.85  
19                 77.78  
436                72.22  
17                 72.00  
438                71.79  
715    

# CFS — Clutch Factor Score (Final: 3 Components)

## Overview
CFS measures match-winning contributions — not just runs scored,
but runs that translate into team success. This complements PPI
(pressure performance) by adding the dimension of result impact.

## Why 3 components, not the originally proposed acceleration metric
An "innings acceleration" idea (SR change within an innings) was
considered but rejected for CFS — it measures batting technique/style,
not match-winning impact. It overlaps with PPI's pressure situations
and is better suited as a future standalone dashboard visualisation,
not a ranking component.

## Components

### Component 1: Win% when scored 50+
Measures general correlation between big scores and match wins.
Threshold: 10+ fifties (75th percentile of fifties_plus distribution).
Qualified batters: 198

### Component 2: Match-winning innings ratio
Was this player the top scorer for their team specifically in wins?
Threshold: 18+ times as top scorer (90th percentile).
Qualified batters: 79

### Component 3: Dominance Score (renamed from Component 4)
Measures individual contribution dominance independent of match result.
Solves the "weak team era" and "milestone player" problem directly —
captures genuine batting greatness even when team result was poor.

Scaling thresholds based on team total:
| Team Total | Threshold |
|---|---|
| < 100 | 30% |
| 100-200 | 35% |
| 200-300 | 40% |
| 300+ | 45% |

Threshold refined from 75th to 90th percentile (63+ innings) to ensure
comparison only between batters with substantial career sample sizes,
avoiding noise from batters with few innings.
Qualified batters: 192

## Why this addresses the original concern
A legendary batter in a weak team era (e.g. Sachin's early India teams)
shows high Dominance Score even in losses, because it measures
contribution share, not match result. This was specifically validated:
Tendulkar ranks #1 in Dominance Score (23.61%) — proving the metric
correctly identifies individual greatness independent of team weakness.

In [68]:
from sklearn.preprocessing import MinMaxScaler

# get all batters who appear in player_innings
all_cfs_batters = pd.DataFrame(
    player_innings['batter'].unique(), columns=['batter']
)

cfs_df = all_cfs_batters.copy()

# merge all 3 components
cfs_df = cfs_df.merge(
    simple_cfs_filtered[['batter', 'win_pct_50plus']],
    on='batter', how='left'
)
cfs_df = cfs_df.merge(
    match_winning_filtered[['batter', 'match_winning_ratio']],
    on='batter', how='left'
)
cfs_df = cfs_df.merge(
    dominance_filtered[['batter', 'dominance_rate']],
    on='batter', how='left'
)

# fill missing with population mean
cfs_df['win_pct_50plus'] = cfs_df['win_pct_50plus'].fillna(
    cfs_df['win_pct_50plus'].mean()
)
cfs_df['match_winning_ratio'] = cfs_df['match_winning_ratio'].fillna(
    cfs_df['match_winning_ratio'].mean()
)
cfs_df['dominance_rate'] = cfs_df['dominance_rate'].fillna(
    cfs_df['dominance_rate'].mean()
)

# normalise each to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))
cfs_df['c1_norm'] = scaler.fit_transform(cfs_df[['win_pct_50plus']])
cfs_df['c2_norm'] = scaler.fit_transform(cfs_df[['match_winning_ratio']])
cfs_df['c3_norm'] = scaler.fit_transform(cfs_df[['dominance_rate']])

# weighted CFS
cfs_df['CFS'] = (
    0.35 * cfs_df['c1_norm'] +
    0.35 * cfs_df['c2_norm'] +
    0.30 * cfs_df['c3_norm']
).round(2)

# apply same career balls qualification as PPI (584 balls, 75th percentile)
cfs_df = cfs_df.merge(total_balls, on='batter')
cfs_df = cfs_df[cfs_df['total_balls'] >= min_career_balls]

cfs_df = cfs_df.sort_values('CFS', ascending=False).reset_index(drop=True)
cfs_df['CFS_rank'] = cfs_df.index + 1

print(f"Total qualified batters: {len(cfs_df)}")
print("\nTop 20 by CFS:")
print(cfs_df[['CFS_rank', 'batter', 'CFS', 'c1_norm', 'c2_norm', 'c3_norm']].head(20))

Total qualified batters: 475

Top 20 by CFS:
    CFS_rank          batter    CFS     c1_norm     c2_norm     c3_norm
0          1       A Symonds  80.89  100.000000  100.000000   36.298179
1          2    AC Gilchrist  77.93   79.359345   75.630621   78.949598
2          3         V Kohli  75.88   70.544316   65.288440   94.451504
3          4         HM Amla  75.60   79.696532   65.365212   82.761542
4          5       Q de Kock  72.95   75.168593   62.480807   82.592122
5          6    SR Tendulkar  72.58   65.522640   56.130730  100.000000
6          7      RT Ponting  71.90   81.707611   83.384514   47.056332
7          8      MJ Guptill  69.95   75.686416   69.061198   64.294790
8          9  AB de Villiers  69.44   72.880539   69.291511   65.607793
9         10       RG Sharma  68.02   67.509634   62.294363   75.307073
10        11       G Gambhir  67.72   67.750482   63.445931   72.681067
11        12     Tamim Iqbal  67.12   57.418112   63.445931   82.719187
12        13   ST J

In [69]:
cfs_df.to_csv(
    r"D:\CricMetric-AI\data\processed\cfs_scores.csv",
    index=False
)
print(f"CFS saved. Total batters: {len(cfs_df)}")

CFS saved. Total batters: 475


## ENS — Era Normalisation Score

### Components
1. Era-adjusted average (built) — accounts for scoring rate by year
2. Career volume score (building now) — accounts for total contribution
3. Opposition quality index (future, needs ICC rankings data)

### Why volume matters
A player with era-adjusted average 55 across 15,000 balls represents
more sustained excellence than the same average across 1,000 balls.
This directly addresses the CJ Anderson vs Kohli concern raised
earlier — PPI alone (rate-based) couldn't distinguish career length,
but ENS explicitly rewards volume.

In [70]:
# average runs per ball by year - this defines "era difficulty"
era_scoring = master_df[master_df['wides']==0].groupby('year').agg(
    total_runs=('runs_off_bat', 'sum'),
    total_balls=('runs_off_bat', 'count')
).reset_index()

era_scoring['runs_per_ball'] = (
    era_scoring['total_runs'] / era_scoring['total_balls']
).round(4)

print(era_scoring)

    year  total_runs  total_balls  runs_per_ball
0   2002        1434         1780         0.8056
1   2003       42754        60564         0.7059
2   2004       30154        40628         0.7422
3   2005       27218        35115         0.7751
4   2006       47211        63702         0.7411
5   2007       60849        77825         0.7819
6   2008       38249        49096         0.7791
7   2009       48693        59762         0.8148
8   2010       45346        56130         0.8079
9   2011       56562        71617         0.7898
10  2012       34928        43747         0.7984
11  2013       51101        63472         0.8051
12  2014       46492        55211         0.8421
13  2015       58619        65876         0.8898
14  2016       38580        44028         0.8763
15  2017       48229        55426         0.8702
16  2018       45543        54283         0.8390
17  2019       58575        67266         0.8708
18  2020       20524        23683         0.8666
19  2021       25861

In [71]:
# normalize era scoring rate around overall mean
overall_mean_rpb = master_df[master_df['wides']==0]['runs_off_bat'].count()
overall_mean_rpb = era_scoring['runs_per_ball'].mean()

era_scoring['era_difficulty_factor'] = (
    overall_mean_rpb / era_scoring['runs_per_ball']
).round(4)

print(f"Overall mean runs/ball: {overall_mean_rpb:.4f}")
print(era_scoring[['year', 'runs_per_ball', 'era_difficulty_factor']])

Overall mean runs/ball: 0.8134
    year  runs_per_ball  era_difficulty_factor
0   2002         0.8056                 1.0097
1   2003         0.7059                 1.1523
2   2004         0.7422                 1.0959
3   2005         0.7751                 1.0494
4   2006         0.7411                 1.0976
5   2007         0.7819                 1.0403
6   2008         0.7791                 1.0440
7   2009         0.8148                 0.9983
8   2010         0.8079                 1.0068
9   2011         0.7898                 1.0299
10  2012         0.7984                 1.0188
11  2013         0.8051                 1.0103
12  2014         0.8421                 0.9659
13  2015         0.8898                 0.9141
14  2016         0.8763                 0.9282
15  2017         0.8702                 0.9347
16  2018         0.8390                 0.9695
17  2019         0.8708                 0.9341
18  2020         0.8666                 0.9386
19  2021         0.7843      

In [72]:
# get balls faced per player per year
player_year_balls = master_df[master_df['wides']==0].groupby(
    ['batter', 'year']
)['runs_off_bat'].count().reset_index()
player_year_balls.columns = ['batter', 'year', 'balls_in_year']

# merge era difficulty factor
player_year_balls = player_year_balls.merge(
    era_scoring[['year', 'era_difficulty_factor']], on='year'
)

# weighted average era factor per player (weighted by balls faced that year)
player_year_balls['weighted_factor'] = (
    player_year_balls['balls_in_year'] * player_year_balls['era_difficulty_factor']
)

player_era_factor = player_year_balls.groupby('batter').agg(
    total_weighted=('weighted_factor', 'sum'),
    total_balls_check=('balls_in_year', 'sum')
).reset_index()

player_era_factor['avg_era_factor'] = (
    player_era_factor['total_weighted'] / player_era_factor['total_balls_check']
).round(4)

print(player_era_factor.head(10))
print(f"\nTotal players: {len(player_era_factor)}")

         batter  total_weighted  total_balls_check  avg_era_factor
0       A Ashok         11.1276                 12          0.9273
1    A Athanaze        501.2889                525          0.9548
2       A Bagai       1019.3653                971          1.0498
3   A Balbirnie       3350.4081               3487          0.9608
4      A Bohara         13.1404                 14          0.9386
5  A Codrington         82.1816                 72          1.1414
6   A Dananjaya        421.9661                441          0.9568
7        A Dutt        353.7360                361          0.9799
8    A Flintoff       2578.7437               2377          1.0849
9      A Flower        528.9057                459          1.1523

Total players: 1897


In [73]:
check_players = ['SR Tendulkar', 'V Kohli', 'BC Lara', 'RT Ponting', 'CJ Anderson','MS Dhoni']
print(player_era_factor[player_era_factor['batter'].isin(check_players)])

            batter  total_weighted  total_balls_check  avg_era_factor
228        BC Lara       2643.8668               2404          1.0998
325    CJ Anderson        951.2099               1012          0.9399
1101      MS Dhoni      11864.4885              11820          1.0038
1444    RT Ponting       8935.4085               8481          1.0536
1584  SR Tendulkar       7868.0344               7414          1.0612
1800       V Kohli      15228.7381              15652          0.9730


In [74]:
# get raw career stats per player
career_stats = master_df[master_df['wides']==0].groupby('batter').agg(
    career_runs=('runs_off_bat', 'sum'),
    career_balls=('runs_off_bat', 'count'),
    career_dismissals=('is_wicket', 'sum')
).reset_index()

career_stats['career_avg'] = (
    career_stats['career_runs'] / career_stats['career_dismissals'].replace(0, 1)
).round(2)

career_stats['career_sr'] = (
    career_stats['career_runs'] / career_stats['career_balls'] * 100
).round(2)

# merge era factor
career_stats = career_stats.merge(player_era_factor[['batter', 'avg_era_factor']], on='batter')

# era-adjusted average
career_stats['era_adj_avg'] = (
    career_stats['career_avg'] * career_stats['avg_era_factor']
).round(2)

print(career_stats[career_stats['batter'].isin(check_players)][
    ['batter', 'career_avg', 'avg_era_factor', 'era_adj_avg', 'career_runs']
])

            batter  career_avg  avg_era_factor  era_adj_avg  career_runs
228        BC Lara       34.78          1.0998        38.25         1878
325    CJ Anderson       27.55          0.9399        25.89         1102
1101      MS Dhoni       47.35          1.0038        47.53        10274
1444    RT Ponting       40.81          1.0536        43.00         6979
1584  SR Tendulkar       48.01          1.0612        50.95         6433
1800       V Kohli       56.88          0.9730        55.34        14675


In [75]:
kohli_check = master_df[(master_df['batter']=='V Kohli') & (master_df['wides']==0)]
print(f"Total runs: {kohli_check['runs_off_bat'].sum()}")
print(f"Total dismissals (is_wicket sum): {kohli_check['is_wicket'].sum()}")
print(f"Calculated avg: {kohli_check['runs_off_bat'].sum() / kohli_check['is_wicket'].sum():.2f}")

# check actual ODI innings count for Kohli
kohli_innings = master_df[master_df['batter']=='V Kohli'].groupby('match_id')['innings'].nunique().sum()
print(f"\nTotal innings batted: {kohli_innings}")

Total runs: 14675
Total dismissals (is_wicket sum): 258
Calculated avg: 56.88

Total innings batted: 296


In [76]:
# check if our data missed some matches/innings due to parser failures
# remember we had 14 failed files earlier
print(f"Kohli runs by year:")
kohli_by_year = master_df[(master_df['batter']=='V Kohli') & (master_df['wides']==0)].groupby('year')['runs_off_bat'].sum()
print(kohli_by_year)
print(f"\nTotal: {kohli_by_year.sum()}")

Kohli runs by year:
year
2008     159
2009     325
2010     995
2011    1381
2012    1026
2013    1268
2014    1054
2015     623
2016     739
2017    1460
2018    1202
2019    1310
2020     431
2021     129
2022     302
2023    1322
2024      58
2025     651
2026     240
Name: runs_off_bat, dtype: int64

Total: 14675


### Data completeness note
Cricsheet coverage starts 2002 and may have small gaps in less
prominent bilateral series. Cross-checking computed averages
against publicly known statistics shows ~2% variance (e.g.
Kohli's computed average of 56.88 vs commonly cited 57-58),
which is within acceptable tolerance for a third-party dataset
of this scale (1.3M+ ball-by-ball records).

In [77]:
# check distribution of career_runs and career_balls
print("Career runs distribution:")
print(career_stats['career_runs'].describe())
print(f"\n75th percentile: {career_stats['career_runs'].quantile(0.75):.0f}")
print(f"90th percentile: {career_stats['career_runs'].quantile(0.90):.0f}")

Career runs distribution:
count     1897.000000
mean       566.352135
std       1269.410845
min          0.000000
25%         23.000000
50%        119.000000
75%        449.000000
max      14675.000000
Name: career_runs, dtype: float64

75th percentile: 449
90th percentile: 1507


In [78]:
import numpy as np

# log transform to handle skewness, then normalize
career_stats['log_runs'] = np.log1p(career_stats['career_runs'])  # log1p handles 0 values safely

scaler_vol = MinMaxScaler(feature_range=(0, 100))
career_stats['volume_norm'] = scaler_vol.fit_transform(career_stats[['log_runs']])

print(career_stats[career_stats['batter'].isin(check_players)][
    ['batter', 'career_runs', 'log_runs', 'volume_norm']
])

            batter  career_runs  log_runs  volume_norm
228        BC Lara         1878  7.538495    78.575355
325    CJ Anderson         1102  7.005789    73.022846
1101      MS Dhoni        10274  9.237469    96.284126
1444    RT Ponting         6979  8.850804    92.253836
1584  SR Tendulkar         6433  8.769352    91.404839
1800       V Kohli        14675  9.593969   100.000000


In [79]:
# get bowling team for each innings (it's the team NOT batting)
innings_teams = master_df.groupby(['match_id', 'innings']).agg(
    batting_team=('batting_team', 'first'),
    team1=('team1', 'first'),
    team2=('team2', 'first')
).reset_index()

innings_teams['bowling_team'] = innings_teams.apply(
    lambda row: row['team2'] if row['batting_team'] == row['team1'] else row['team1'],
    axis=1
)

print(innings_teams.head(10))

  match_id  innings batting_team      team1     team2 bowling_team
0  1000887        1    Australia  Australia  Pakistan     Pakistan
1  1000887        2     Pakistan  Australia  Pakistan    Australia
2  1000889        1    Australia  Australia  Pakistan     Pakistan
3  1000889        2     Pakistan  Australia  Pakistan    Australia
4  1000891        1     Pakistan  Australia  Pakistan    Australia
5  1000891        2    Australia  Australia  Pakistan     Pakistan
6  1000893        1    Australia  Australia  Pakistan     Pakistan
7  1000893        2     Pakistan  Australia  Pakistan    Australia
8  1000895        1    Australia  Australia  Pakistan     Pakistan
9  1000895        2     Pakistan  Australia  Pakistan    Australia


In [80]:
# merge bowling team info into master_df
master_with_bowling = master_df.merge(
    innings_teams[['match_id', 'innings', 'bowling_team']],
    on=['match_id', 'innings']
)

# team bowling economy by year (lower = stronger bowling)
team_bowling_economy = master_with_bowling[master_with_bowling['wides']==0].groupby(
    ['bowling_team', 'year']
).agg(
    runs_conceded=('runs_off_bat', 'sum'),
    balls_bowled=('runs_off_bat', 'count')
).reset_index()

team_bowling_economy['economy_rate'] = (
    team_bowling_economy['runs_conceded'] / team_bowling_economy['balls_bowled'] * 6
).round(3)

print(team_bowling_economy.head(10))
print(f"\nOverall economy distribution:")
print(team_bowling_economy['economy_rate'].describe())

  bowling_team  year  runs_conceded  balls_bowled  economy_rate
0    Africa XI  2005            180           312         3.462
1    Africa XI  2007            920           908         6.079
2      Asia XI  2005            269           470         3.434
3      Asia XI  2007            865           891         5.825
4    Australia  2003           4715          7157         3.953
5    Australia  2004           3926          5531         4.259
6    Australia  2005           3091          4336         4.277
7    Australia  2006           5534          7331         4.529
8    Australia  2007           6380          8318         4.602
9    Australia  2008           2997          4498         3.998

Overall economy distribution:
count    388.000000
mean       4.858652
std        0.602999
min        2.969000
25%        4.473750
50%        4.879500
75%        5.284000
max        6.823000
Name: economy_rate, dtype: float64


In [81]:
print("Balls bowled distribution (year-wise):")
print(team_bowling_economy['balls_bowled'].describe())
print(f"\n25th percentile: {team_bowling_economy['balls_bowled'].quantile(0.25):.0f}")
print(f"Median: {team_bowling_economy['balls_bowled'].median():.0f}")

Balls bowled distribution (year-wise):
count      388.000000
mean      3397.677835
std       2305.181101
min        123.000000
25%       1382.750000
50%       3038.500000
75%       5037.750000
max      10213.000000
Name: balls_bowled, dtype: float64

25th percentile: 1383
Median: 3038


In [82]:
min_balls_bowled = int(team_bowling_economy['balls_bowled'].quantile(0.25))
print(f"Minimum balls bowled threshold (25th percentile): {min_balls_bowled}")

team_bowling_filtered = team_bowling_economy[
    team_bowling_economy['balls_bowled'] >= min_balls_bowled
].copy()

print(f"Team-year combinations before filter: {len(team_bowling_economy)}")
print(f"Team-year combinations after filter: {len(team_bowling_filtered)}")

# now rank teams by economy within this filtered set
# lower economy = stronger bowling = higher opposition quality score
scaler_opp = MinMaxScaler(feature_range=(0, 100))

# invert economy so LOWER economy gets HIGHER score
team_bowling_filtered['opposition_quality'] = 100 - scaler_opp.fit_transform(
    team_bowling_filtered[['economy_rate']]
)

print("\nSample opposition quality scores:")
print(team_bowling_filtered[['bowling_team', 'year', 'economy_rate', 'opposition_quality']].sort_values('opposition_quality', ascending=False).head(15))

Minimum balls bowled threshold (25th percentile): 1382
Team-year combinations before filter: 388
Team-year combinations after filter: 292

Sample opposition quality scores:
                 bowling_team  year  economy_rate  opposition_quality
183               Netherlands  2021         3.847          100.000000
31                 Bangladesh  2006         3.941           95.886214
69                    England  2003         3.945           95.711160
186               Netherlands  2024         3.945           95.711160
4                   Australia  2003         3.953           95.361050
70                    England  2004         3.971           94.573304
101                     India  2003         3.982           94.091904
219                      Oman  2024         3.990           93.741794
9                   Australia  2008         3.998           93.391685
333  United States of America  2019         4.019           92.472648
160                   Namibia  2022         4.038        

In [83]:
netherlands_check = team_bowling_filtered[
    (team_bowling_filtered['bowling_team']=='Netherlands') & 
    (team_bowling_filtered['year']==2021)
]
print(netherlands_check)

# also check how many matches this represents
netherlands_matches = master_with_bowling[
    (master_with_bowling['bowling_team']=='Netherlands') & 
    (master_with_bowling['year']==2021)
]['match_id'].nunique()
print(f"\nNumber of matches: {netherlands_matches}")

    bowling_team  year  runs_conceded  balls_bowled  economy_rate  \
183  Netherlands  2021           1031          1608         3.847   

     opposition_quality  
183               100.0  

Number of matches: 6


In [84]:
min_balls_bowled_mean = int(team_bowling_economy['balls_bowled'].quantile(0.25))
print(f"Minimum balls bowled threshold (mean): {min_balls_bowled_mean}")

team_bowling_filtered = team_bowling_economy[
    team_bowling_economy['balls_bowled'] >= min_balls_bowled_mean
].copy()

print(f"Team-year combinations before filter: {len(team_bowling_economy)}")
print(f"Team-year combinations after filter: {len(team_bowling_filtered)}")

scaler_opp = MinMaxScaler(feature_range=(0, 100))
team_bowling_filtered['opposition_quality'] = 100 - scaler_opp.fit_transform(
    team_bowling_filtered[['economy_rate']]
)

print("\nTop 15 by opposition quality (toughest bowling attacks):")
print(team_bowling_filtered[['bowling_team', 'year', 'economy_rate', 'opposition_quality']].sort_values('opposition_quality', ascending=False).head(15))

Minimum balls bowled threshold (mean): 1382
Team-year combinations before filter: 388
Team-year combinations after filter: 292

Top 15 by opposition quality (toughest bowling attacks):
                 bowling_team  year  economy_rate  opposition_quality
183               Netherlands  2021         3.847          100.000000
31                 Bangladesh  2006         3.941           95.886214
69                    England  2003         3.945           95.711160
186               Netherlands  2024         3.945           95.711160
4                   Australia  2003         3.953           95.361050
70                    England  2004         3.971           94.573304
101                     India  2003         3.982           94.091904
219                      Oman  2024         3.990           93.741794
9                   Australia  2008         3.998           93.391685
333  United States of America  2019         4.019           92.472648
160                   Namibia  2022         4

In [85]:
team_bowling_full = master_with_bowling[master_with_bowling['wides']==0].groupby(
    ['bowling_team', 'year']
).agg(
    runs_conceded=('runs_off_bat', 'sum'),
    balls_bowled=('runs_off_bat', 'count'),
    wickets_taken=('is_wicket', 'sum')
).reset_index()

team_bowling_full['economy_rate'] = (
    team_bowling_full['runs_conceded'] / team_bowling_full['balls_bowled'] * 6
).round(3)

team_bowling_full['bowling_sr'] = (
    team_bowling_full['balls_bowled'] / team_bowling_full['wickets_taken'].replace(0,1)
).round(2)
print(team_bowling_full.head(10))

  bowling_team  year  runs_conceded  balls_bowled  wickets_taken  \
0    Africa XI  2005            180           312             12   
1    Africa XI  2007            920           908             24   
2      Asia XI  2005            269           470             20   
3      Asia XI  2007            865           891             27   
4    Australia  2003           4715          7157            231   
5    Australia  2004           3926          5531            171   
6    Australia  2005           3091          4336            144   
7    Australia  2006           5534          7331            225   
8    Australia  2007           6380          8318            268   
9    Australia  2008           2997          4498            154   

   economy_rate  bowling_sr  
0         3.462       26.00  
1         6.079       37.83  
2         3.434       23.50  
3         5.825       33.00  
4         3.953       30.98  
5         4.259       32.35  
6         4.277       30.11  
7         4

In [86]:

print(f"\nBowling SR distribution:")
print(team_bowling_full['bowling_sr'].describe())


Bowling SR distribution:
count    388.000000
mean      37.086340
std        8.155207
min       19.200000
25%       32.642500
50%       35.730000
75%       39.925000
max      123.000000
Name: bowling_sr, dtype: float64


In [87]:
team_bowling_filtered_full = team_bowling_full[
    team_bowling_full['balls_bowled'] >= min_balls_bowled
].copy()

print(f"Team-year combinations after filter: {len(team_bowling_filtered_full)}")

# normalize both metrics - lower is better for both, so invert
scaler_econ = MinMaxScaler(feature_range=(0, 100))
scaler_sr = MinMaxScaler(feature_range=(0, 100))

team_bowling_filtered_full['economy_score'] = 100 - scaler_econ.fit_transform(
    team_bowling_filtered_full[['economy_rate']]
)
team_bowling_filtered_full['strike_rate_score'] = 100 - scaler_sr.fit_transform(
    team_bowling_filtered_full[['bowling_sr']]
)

# combine - equal weight for now
team_bowling_filtered_full['opposition_quality'] = (
    0.5 * team_bowling_filtered_full['economy_score'] +
    0.5 * team_bowling_filtered_full['strike_rate_score']
).round(2)

print("\nTop 15 by combined opposition quality:")
print(team_bowling_filtered_full[
    ['bowling_team', 'year', 'economy_rate', 'bowling_sr', 'opposition_quality']
].sort_values('opposition_quality', ascending=False).head(15))

Team-year combinations after filter: 292

Top 15 by combined opposition quality:
                 bowling_team  year  economy_rate  bowling_sr  \
220                      Oman  2025         4.038       27.84   
186               Netherlands  2024         3.945       29.46   
333  United States of America  2019         4.019       28.25   
267                  Scotland  2024         4.038       28.52   
9                   Australia  2008         3.998       29.21   
4                   Australia  2003         3.953       30.98   
339  United States of America  2025         4.209       28.35   
298                 Sri Lanka  2007         4.127       29.80   
70                    England  2004         3.971       32.52   
219                      Oman  2024         3.990       32.42   
278              South Africa  2011         4.281       27.95   
264                  Scotland  2021         4.124       30.71   
191               New Zealand  2004         4.049       32.05   
216      

In [88]:
# check Namibia 2022
namibia_check = master_with_bowling[
    (master_with_bowling['bowling_team']=='Namibia') & 
    (master_with_bowling['year']==2022)
]
print("Namibia 2022 opponents:")
print(namibia_check['batting_team'].unique())
print(f"Matches: {namibia_check['match_id'].nunique()}")
print(f"Balls bowled: {len(namibia_check[namibia_check['wides']==0])}")

# check UAE 2022
uae_check = master_with_bowling[
    (master_with_bowling['bowling_team']=='United Arab Emirates') & 
    (master_with_bowling['year']==2022)
]
print("\nUAE 2022 opponents:")
print(uae_check['batting_team'].unique())
print(f"Matches: {uae_check['match_id'].nunique()}")
print(f"Balls bowled: {len(uae_check[uae_check['wides']==0])}")

Namibia 2022 opponents:
<ArrowStringArray>
[                    'Oman',     'United Arab Emirates',
                    'Nepal',                 'Scotland',
 'United States of America',         'Papua New Guinea']
Length: 6, dtype: str
Matches: 20
Balls bowled: 5482

UAE 2022 opponents:
<ArrowStringArray>
[                    'Oman',                  'Namibia',
         'Papua New Guinea',                    'Nepal',
                 'Scotland', 'United States of America']
Length: 6, dtype: str
Matches: 21
Balls bowled: 5882


In [89]:
full_members = [
    'Australia', 'England', 'India', 'Pakistan', 'South Africa',
    'New Zealand', 'Sri Lanka', 'West Indies', 'Bangladesh',
    'Zimbabwe', 'Afghanistan', 'Ireland'
]

team_bowling_filtered_full2 = team_bowling_filtered_full[
    team_bowling_filtered_full['bowling_team'].isin(full_members)
].copy()

print(f"Team-year combinations after Full Member filter: {len(team_bowling_filtered_full2)}")
print("\nTop 15 by combined opposition quality (Full Members only):")
print(team_bowling_filtered_full2[
    ['bowling_team', 'year', 'economy_rate', 'bowling_sr', 'opposition_quality']
].sort_values('opposition_quality', ascending=False).head(25))

Team-year combinations after Full Member filter: 233

Top 15 by combined opposition quality (Full Members only):
     bowling_team  year  economy_rate  bowling_sr  opposition_quality
9       Australia  2008         3.998       29.21               90.09
4       Australia  2003         3.953       30.98               88.81
298     Sri Lanka  2007         4.127       29.80               86.51
70        England  2004         3.971       32.52               86.44
278  South Africa  2011         4.281       27.95               85.51
191   New Zealand  2004         4.049       32.05               85.33
101         India  2003         3.982       33.82               84.53
69        England  2003         3.945       35.49               83.20
6       Australia  2005         4.277       30.11               82.83
230      Pakistan  2011         4.096       33.26               82.76
31     Bangladesh  2006         3.941       36.27               82.29
299     Sri Lanka  2008         4.268       31.

In [90]:
team_bowling_filtered_full2[team_bowling_filtered_full2['year']==2023].sort_values('opposition_quality', ascending=False)

,bowling_team,year,runs_conceded,balls_bowled,wickets_taken,economy_rate,bowling_sr,economy_score,strike_rate_score,opposition_quality
121,India,2023,6249,7648,287,4.902,26.65,53.829322,93.340164,73.58
314,Sri Lanka,2023,5499,6130,201,5.382,30.50,32.822757,83.478484,58.15
290,South Africa,2023,5586,6106,206,5.489,29.64,28.140044,85.681352,56.91
385,Zimbabwe,2023,3152,3795,98,4.983,38.72,50.284464,62.423156,56.35
361,West Indies,2023,4129,4690,136,5.282,34.49,37.199125,73.258197,55.23
89,England,2023,5205,5667,174,5.511,32.57,27.177243,78.176230,52.68
142,Ireland,2023,4554,4946,148,5.524,33.42,26.608315,75.998975,51.30
24,Australia,2023,5308,5731,173,5.557,33.13,25.164114,76.741803,50.95
48,Bangladesh,2023,5759,6471,171,5.340,37.84,34.660832,64.677254,49.67
242,Pakistan,2023,5441,5753,174,5.675,33.06,20.000000,76.921107,48.46


### Opposition Quality Index — Full Member Restriction

### Problem discovered
Initial team-year bowling economy + strike rate calculation showed
Namibia (2022) and UAE (2022) ranking in the global top 10 for
opposition quality, despite not being elite bowling attacks.

### Root cause
Investigation revealed both teams' 2022 matches were exclusively
against other ICC Associate Members (Oman, Nepal, Scotland, USA,
Papua New Guinea) — never against Full Member Test-playing nations.
Their strong economy/strike rate numbers reflected weak opposition
batting, not genuine bowling quality.

### Fix
Restricted opposition_quality calculation to the 12 ICC Full Member
nations only (Australia, England, India, Pakistan, South Africa,
New Zealand, Sri Lanka, West Indies, Bangladesh, Zimbabwe,
Afghanistan, Ireland). This ensures opposition quality scores
reflect genuine bowling strength against credible competition,
not inflated stats from associate-vs-associate matches.

### Validation
Post-fix top 15 rankings align with well-known historically strong
bowling eras: Australia 2003-2008 (McGrath/Lee era), South Africa
2011 (Steyn/Morkel), Sri Lanka 2007-2008 (Murali/Malinga).

In [91]:
player_innings_with_opp = player_innings.merge(
    master_df[['match_id','innings','year']].drop_duplicates(),
    on=['match_id','innings']
)

player_innings_with_opp = player_innings_with_opp.merge(
    innings_teams[['match_id','innings','bowling_team']],
    on=['match_id','innings']
)

player_innings_with_opp = player_innings_with_opp.merge(
    team_bowling_filtered_full2[['bowling_team','year','opposition_quality']],
    on=['bowling_team','year'], how='left'
)

print(f"Total records: {len(player_innings_with_opp)}")
print(f"Records with opposition quality matched: {player_innings_with_opp['opposition_quality'].notna().sum()}")
print(f"Match rate: {player_innings_with_opp['opposition_quality'].notna().mean()*100:.1f}%")

Total records: 44409
Records with opposition quality matched: 36265
Match rate: 81.7%


In [92]:
check_missing = player_innings_with_opp[player_innings_with_opp['opposition_quality'].isna()]
print("Bowling teams with missing opposition_quality:")
print(check_missing['bowling_team'].value_counts().head(15))

Bowling teams with missing opposition_quality:
bowling_team
Scotland                    1057
United Arab Emirates         982
Netherlands                  785
Nepal                        763
United States of America     707
Oman                         663
Namibia                      646
Canada                       554
Papua New Guinea             484
Kenya                        316
Ireland                      253
Hong Kong                    156
Bermuda                       95
New Zealand                   90
Pakistan                      85
Name: count, dtype: int64


In [93]:
# check why Ireland is missing despite being Full Member
ireland_check = team_bowling_filtered_full2[
    team_bowling_filtered_full2['bowling_team']=='Ireland'
]
print("Ireland entries in filtered table:")
print(ireland_check)

# check which years Ireland/NZ/Pakistan are missing
nz_missing = master_with_bowling[
    (master_with_bowling['bowling_team']=='New Zealand')
]['year'].unique()
print(f"\nAll New Zealand years in data: {sorted(nz_missing)}")

nz_in_filtered = team_bowling_filtered_full2[
    team_bowling_filtered_full2['bowling_team']=='New Zealand'
]['year'].unique()
print(f"New Zealand years that PASSED filter: {sorted(nz_in_filtered)}")

Ireland entries in filtered table:
    bowling_team  year  runs_conceded  balls_bowled  wickets_taken  \
126      Ireland  2007           1768          2384             58   
130      Ireland  2011           2061          2599             76   
134      Ireland  2015           2781          3124             73   
136      Ireland  2017           1710          1916             48   
137      Ireland  2018           1906          2556             85   
138      Ireland  2019           1792          1981             47   
139      Ireland  2020           1279          1383             39   
140      Ireland  2021           1793          2481             71   
141      Ireland  2022           1472          1683             52   
142      Ireland  2023           4554          4946            148   
144      Ireland  2025           1608          1646             41   

     economy_rate  bowling_sr  economy_score  strike_rate_score  \
126         4.450       41.10      73.610503          56.

In [94]:
# fill missing opposition_quality with population mean
opp_mean = player_innings_with_opp['opposition_quality'].mean()
print(f"Population mean opposition quality: {opp_mean:.2f}")

player_innings_with_opp['opposition_quality'] = player_innings_with_opp[
    'opposition_quality'
].fillna(opp_mean)

print(f"Missing after fill: {player_innings_with_opp['opposition_quality'].isna().sum()}")

# now compute per batter weighted opposition quality
# weighted by runs scored (more credit for runs against tougher teams)
player_innings_with_opp['opp_weighted_runs'] = (
    player_innings_with_opp['player_runs'] * 
    player_innings_with_opp['opposition_quality']
)

batter_opp_quality = player_innings_with_opp.groupby('batter').agg(
    total_opp_weighted=('opp_weighted_runs', 'sum'),
    total_runs=('player_runs', 'sum')
).reset_index()

batter_opp_quality['avg_opp_quality'] = (
    batter_opp_quality['total_opp_weighted'] / 
    batter_opp_quality['total_runs'].replace(0, 1)
).round(2)

print(f"\nOpposition quality distribution:")
print(batter_opp_quality['avg_opp_quality'].describe())
print(f"\nCheck players:")
print(batter_opp_quality[batter_opp_quality['batter'].isin(check_players)][
    ['batter', 'avg_opp_quality']
])

Population mean opposition quality: 61.05
Missing after fill: 0

Opposition quality distribution:
count    1897.000000
mean       59.930322
std        12.779476
min         0.000000
25%        56.480000
50%        61.050000
75%        65.570000
max        90.090000
Name: avg_opp_quality, dtype: float64

Check players:
            batter  avg_opp_quality
228        BC Lara            67.68
325    CJ Anderson            52.67
1101      MS Dhoni            60.01
1444    RT Ponting            65.66
1584  SR Tendulkar            68.41
1800       V Kohli            53.89


In [95]:
# recompute era_adj_avg_norm
scaler_era = MinMaxScaler(feature_range=(0, 100))
career_stats['era_adj_avg_norm'] = scaler_era.fit_transform(
    career_stats[['era_adj_avg']]
)

# now build ENS df correctly
ens_df = career_stats[['batter', 'career_runs', 'career_balls',
                        'career_avg', 'era_adj_avg',
                        'era_adj_avg_norm', 'volume_norm']].copy()

# merge opposition quality
ens_df = ens_df.merge(
    batter_opp_quality[['batter', 'avg_opp_quality']],
    on='batter', how='left'
)

# normalize opposition quality
scaler_ens = MinMaxScaler(feature_range=(0, 100))
ens_df['opp_quality_norm'] = scaler_ens.fit_transform(
    ens_df[['avg_opp_quality']]
)

# final ENS
ens_df['ENS'] = (
    0.40 * ens_df['era_adj_avg_norm'] +
    0.30 * ens_df['volume_norm'] +
    0.30 * ens_df['opp_quality_norm']
).round(2)

# apply career balls qualification
ens_df = ens_df.merge(total_balls[['batter','total_balls']], on='batter')
ens_df = ens_df[ens_df['total_balls'] >= min_career_balls]

ens_df = ens_df.sort_values('ENS', ascending=False).reset_index(drop=True)
ens_df['ENS_rank'] = ens_df.index + 1

print(f"Total qualified batters: {len(ens_df)}")
print("\nTop 20 by ENS:")
print(ens_df[['ENS_rank', 'batter', 'ENS',
              'era_adj_avg_norm', 'volume_norm',
              'opp_quality_norm']].head(20))

Total qualified batters: 475

Top 20 by ENS:
    ENS_rank          batter    ENS  era_adj_avg_norm  volume_norm  \
0          1    SR Tendulkar  70.40         50.490536    91.404839   
1          2         V Kohli  69.88         54.840947   100.000000   
2          3  AB de Villiers  69.88         54.741849    95.396260   
3          4   KC Sangakkara  68.75         46.110395    97.585148   
4          5       ML Hayden  68.52         50.777921    86.293130   
5          6        MS Dhoni  67.71         47.101377    96.284126   
6          7       IJL Trott  67.15         51.778813    82.807150   
7          8      JA Rudolph  66.62         53.156278    69.945226   
8          9       JH Kallis  66.60         46.368051    88.629580   
9         10      RT Ponting  66.59         42.612229    92.253836   
10        11       MJ Clarke  66.27         44.148251    92.267267   
11        12    AC Gilchrist  66.19         42.552770    87.339345   
12        13       RG Sharma  66.10         4

In [96]:
# check what columns career_stats actually has
print(career_stats.columns.tolist())

['batter', 'career_runs', 'career_balls', 'career_dismissals', 'career_avg', 'career_sr', 'avg_era_factor', 'era_adj_avg', 'log_runs', 'volume_norm', 'era_adj_avg_norm']


In [97]:
ens_df.to_csv(
    r"D:\CricMetric-AI\data\processed\ens_scores.csv",
    index=False
)
print(f"ENS saved. Total batters: {len(ens_df)}")

ENS saved. Total batters: 475


## Master Player Feature Matrix

### Overview
Combines all 3 computed metrics into one unified table.
475 qualified batters, all with scores on every component.

### Final Score Architecture
| Component | Weight | Measures |
|---|---|---|
| PPI | 35% | Pressure performance across 4 situations |
| CFS | 35% | Match-winning clutch contributions |
| ENS | 30% | Era-adjusted overall career quality |

### Why these weights
PPI and CFS equally weighted at 35% each — both measure
different dimensions of "performing when it matters"
ENS slightly lower at 30% — captures baseline quality
but shouldn't dominate over situational excellence

In [98]:
# combine all 3 into master feature matrix
master_features = ppi_df[['batter', 'PPI', 's1_norm', 's2_norm', 
                           's3_norm', 's3_con_norm', 's4_norm']].copy()

master_features = master_features.merge(
    cfs_df[['batter', 'CFS', 'c1_norm', 'c2_norm', 'c3_norm']], 
    on='batter', how='inner'
)

master_features = master_features.merge(
    ens_df[['batter', 'ENS', 'era_adj_avg_norm', 
            'volume_norm', 'opp_quality_norm',
            'career_runs', 'career_avg', 'era_adj_avg']], 
    on='batter', how='inner'
)

print(f"Total batters in master matrix: {len(master_features)}")
print(f"Total features: {master_features.shape[1]}")
print(f"\nColumns: {master_features.columns.tolist()}")

Total batters in master matrix: 475
Total features: 18

Columns: ['batter', 'PPI', 's1_norm', 's2_norm', 's3_norm', 's3_con_norm', 's4_norm', 'CFS', 'c1_norm', 'c2_norm', 'c3_norm', 'ENS', 'era_adj_avg_norm', 'volume_norm', 'opp_quality_norm', 'career_runs', 'career_avg', 'era_adj_avg']


In [99]:
# final combined score
master_features['FINAL_SCORE'] = (
    0.35 * master_features['PPI'] +
    0.35* master_features['CFS'] +
    0.30* master_features['ENS']
).round(2)

# rank
master_features = master_features.sort_values(
    'FINAL_SCORE', ascending=False
).reset_index(drop=True)
master_features['FINAL_RANK'] = master_features.index + 1

print(f"Top 20 ODI Batters — CricMetric-AI Final Rankings:")
print(master_features[['FINAL_RANK', 'batter', 'FINAL_SCORE', 
                        'PPI', 'CFS', 'ENS']].head(20))

Top 20 ODI Batters — CricMetric-AI Final Rankings:
    FINAL_RANK          batter  FINAL_SCORE    PPI    CFS    ENS
0            1         V Kohli        70.15  64.66  75.88  69.88
1            2       A Symonds        66.20  51.73  80.89  65.96
2            3  AB de Villiers        64.90  56.08  69.44  69.88
3            4       RG Sharma        62.04  52.58  68.02  66.10
4            5      RT Ponting        61.02  45.37  71.90  66.59
5            6    AC Gilchrist        60.99  39.59  77.93  66.19
6            7    SR Tendulkar        59.84  38.05  72.58  70.40
7            8         HM Amla        59.55  38.36  75.60  65.54
8            9    Yuvraj Singh        58.98  50.50  61.37  66.09
9           10   KC Sangakkara        58.89  50.63  58.70  68.75
10          11       ML Hayden        58.14  49.94  57.43  68.52
11          12        HH Gibbs        57.88  43.80  67.04  63.61
12          13      Babar Azam        57.87  43.35  66.42  64.85
13          14      TM Dilshan        5

In [100]:
symonds = master_features[master_features['batter'] == 'A Symonds']
print(symonds[['batter', 'FINAL_SCORE', 'PPI', 'CFS', 'ENS',
               'c1_norm', 'c2_norm', 'c3_norm']].to_string())

# check raw CFS components
print("\nCFS components:")
print(simple_cfs_filtered[simple_cfs_filtered['batter']=='A Symonds'])
print(match_winning_filtered[match_winning_filtered['batter']=='A Symonds'])
print(dominance_filtered[dominance_filtered['batter']=='A Symonds'])

      batter  FINAL_SCORE    PPI    CFS    ENS  c1_norm  c2_norm    c3_norm
1  A Symonds         66.2  51.73  80.89  65.96    100.0    100.0  36.298179

CFS components:
       batter  fifties_plus  wins_with_50plus  win_pct_50plus
11  A Symonds            29                28           96.55
      batter  times_top_scorer  times_topscore_in_win  match_winning_ratio
9  A Symonds                22                     22                100.0
       batter  total_innings  dominant_innings  dominance_rate
30  A Symonds            105                 9            8.57


In [101]:
# check Australia's overall win rate in our dataset
aus_wins = match_results[
    (match_results['winner']=='Australia')
].shape[0]
aus_matches = master_df[
    (master_df['team1']=='Australia') | 
    (master_df['team2']=='Australia')
]['match_id'].nunique()

print(f"Australia win rate: {aus_wins/aus_matches*100:.1f}%")

# compare to Symonds' match winning ratio
print(f"Symonds match-winning ratio: 100%")
print(f"If Australia wins X% overall, Symonds' true clutch above baseline = 100 - X%")

Australia win rate: 63.1%
Symonds match-winning ratio: 100%
If Australia wins X% overall, Symonds' true clutch above baseline = 100 - X%


In [102]:
# compute each team's overall win rate
team_win_rates = pd.DataFrame()

for team in master_df['team1'].unique():
    team_matches = match_results[
        (match_results['team_innings1']==team) | 
        (match_results['team_innings2']==team)
    ]
    wins = match_results[match_results['winner']==team].shape[0]
    total = len(team_matches)
    if total > 0:
        team_win_rates = pd.concat([team_win_rates, pd.DataFrame({
            'team': [team],
            'team_win_rate': [wins/total*100]
        })])

team_win_rates = team_win_rates.reset_index(drop=True)
print(team_win_rates.sort_values('team_win_rate', ascending=False).head(15))

                        team  team_win_rate
0                  Australia      63.888889
6                      India      60.658915
5               South Africa      58.701299
3                New Zealand      55.468750
18  United States of America      51.948052
7                   Pakistan      51.435407
1                   Scotland      51.304348
19                     Nepal      50.000000
12                 Sri Lanka      49.372385
17                      Oman      49.275362
4                    England      48.853211
15                   Namibia      45.070423
8                 Bangladesh      41.049383
22                 Africa XI      40.000000
10               West Indies      39.322917


In [103]:
# map each batter's primary team
batter_team = master_df.groupby('batter')['batting_team'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
batter_team.columns = ['batter', 'primary_team']

# merge team win rate
batter_team = batter_team.merge(
    team_win_rates, left_on='primary_team', right_on='team', how='left'
)
batter_team['team_win_rate'] = batter_team['team_win_rate'].fillna(50.0)

# merge into match_winning_stats
match_winning_adj = match_winning_filtered.merge(
    batter_team[['batter', 'primary_team', 'team_win_rate']],
    on='batter', how='left'
)

# baseline adjusted ratio
match_winning_adj['adj_match_winning_ratio'] = (
    match_winning_adj['match_winning_ratio'] - 
    match_winning_adj['team_win_rate']
).round(2)

print("Check players:")
check_batters = ['A Symonds', 'V Kohli', 'SR Tendulkar', 'RT Ponting', 'BC Lara']
print(match_winning_adj[match_winning_adj['batter'].isin(check_batters)][
    ['batter', 'primary_team', 'team_win_rate', 
     'match_winning_ratio', 'adj_match_winning_ratio']
])

Check players:
          batter primary_team  team_win_rate  match_winning_ratio  \
0      A Symonds    Australia      63.888889               100.00   
55    RT Ponting    Australia      63.888889                84.85   
62  SR Tendulkar        India      60.658915                60.00   
73       V Kohli        India      60.658915                68.35   

    adj_match_winning_ratio  
0                     36.11  
55                    20.96  
62                    -0.66  
73                     7.69  


In [104]:
# rebuild match_winning_filtered with adjusted ratio
match_winning_adj_filtered = match_winning_adj.copy()

# normalize adjusted ratio (can be negative, so shift first)
min_adj = match_winning_adj_filtered['adj_match_winning_ratio'].min()
match_winning_adj_filtered['adj_ratio_shifted'] = (
    match_winning_adj_filtered['adj_match_winning_ratio'] - min_adj
)

scaler_c2 = MinMaxScaler(feature_range=(0, 100))
match_winning_adj_filtered['c2_norm_adj'] = scaler_c2.fit_transform(
    match_winning_adj_filtered[['adj_ratio_shifted']]
)

print("Adjusted Component 2 — Top 10:")
print(match_winning_adj_filtered.sort_values(
    'c2_norm_adj', ascending=False
)[['batter', 'primary_team', 'team_win_rate', 
   'match_winning_ratio', 'adj_match_winning_ratio', 
   'c2_norm_adj']].head(10))

Adjusted Component 2 — Top 10:
           batter primary_team  team_win_rate  match_winning_ratio  \
0       A Symonds    Australia      63.888889               100.00   
14      CR Ervine     Zimbabwe      22.262774                50.00   
72    Tamim Iqbal   Bangladesh      41.049383                66.67   
70     TM Dilshan    Sri Lanka      49.372385                70.83   
55     RT Ponting    Australia      63.888889                84.85   
68  Sikandar Raza     Zimbabwe      22.262774                42.11   
28        IR Bell      England      48.853211                68.00   
77    Younis Khan     Pakistan      51.435407                68.42   
41     MJ Guptill  New Zealand      55.468750                71.79   
13       CH Gayle  West Indies      39.322917                53.66   

    adj_match_winning_ratio  c2_norm_adj  
0                     36.11   100.000000  
14                    27.74    85.156943  
72                    25.62    81.397411  
70                    21.4